<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_11_Hybrid_Neyman_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 11 — A response-matched, simulator-calibrated Neyman construction

Exercise 5 ended with pseudo-experiments from the learned hNDE likelihood. Here we use that likelihood as an ordering model, correct its leading simulator response mismatch, amortize the resulting sampling law over the physical signal strength $\mu$, and calibrate the remaining discrepancy with simulator pseudo-experiments.

The preceding 250,000-event run was an important diagnostic: it achieved $95.026\%$ coverage when calibration and audit used the same empirical template, but only $89.909\%$ on an independently generated 250,000-event template. The PIT layer changed that external result by only $-0.075$ percentage points. That pattern identifies finite event-template transfer—especially its projection along the likelihood score—as the dominant limitation.

This version therefore separates seven statistical objects:

1. the **frozen event-level hNDE likelihood** $L_{\rm H}(\nu)$, whose internal signal-strength coordinate is denoted $\nu$;
2. a fresh, streamed **construction simulator law** with nested post-selection prefixes of 250k, 1M, 4M, 16M, and 64M events per process;
3. a monotone **pseudo-truth response map** $g(\mu)$ from physical simulator truth to the KL-optimal hNDE coordinate, rebuilt from the 64M construction law;
4. the response-corrected ordering statistic $t_\mu^{(g)}=-2\log[L_{\rm H}(g(\mu))/L_{\rm H}(\widehat\nu)]$;
5. a conditional spline plus a first density-ratio correction trained on response-matched hNDE toys;
6. a PIT-space simulator correction trained on the 64M construction law, with extra proposal density at low $\mu$;
7. a genuinely new, independently seeded 64M-event-per-process audit law that remains unopened until the response, neural models, quadratures, and cutoffs have been frozen.

The response map removes the large pseudo-truth displacement but does not assert that the hNDE likelihood is correct. Sandwich-variance changes, skewness, discreteness, and tail differences remain for the PIT ratio to learn. Exact coverage is assessed only after all maps and cutoffs have been frozen.

The expensive event networks are loaded from their Exercise 5 checkpoints. Raw simulator batches are immediately reduced to the 512-bin likelihood-ratio template and feature histograms; no 64M-event feature array is retained. Atomic caches make the billion-scale construction-plus-audit stream resumable across multiple Colab sessions. All response-dependent caches use new versioned paths because the old statistic flow, ratios, endpoint toys, auditor, and inversion are statistically incompatible with the 64M construction.


In [ ]:
## ==========================================================================
# Google Colab setup — run me first in a fresh/restarted runtime.
# ==========================================================================
import os, sys

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
N_BKG, N_SIG = 100_000_000, 20_000_000
USE_DRIVE = True
REMAKE_EVENTS = False

import subprocess
from pathlib import Path

DEPENDENCIES = [
    "pytorch-lightning",
    "onnx",
    "onnxruntime-gpu",
    "onnxscript",
    "iminuit",
    "mplhep",
    "nflows",
    "pyarrow",
]


def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"

    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        #run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)

    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    # The notebook may be open from GitHub while the Drive checkout has
    # an older HEAD or a locally saved notebook. Refresh the full helper
    # dependency set without touching the open notebook.
    run(
        "git", "-C", REPO_DIR, "checkout", f"origin/{BRANCH}", "--",
        "src",
        "workshops/ml4hep_tifr_colab/generate_distributions.py",
        "workshops/ml4hep_tifr_colab/utils.py",
        "workshops/ml4hep_tifr_colab/utils_distributions.py",
        "workshops/ml4hep_tifr_colab/utils_hnpe.py",
        "workshops/ml4hep_tifr_colab/utils_neyman.py",
        "workshops/ml4hep_tifr_colab/utils_nf.py",
        "workshops/ml4hep_tifr_colab/utils_plotting.py",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)
    from importlib import metadata as importlib_metadata
    def distribution_is_installed(name):
        try:
            importlib_metadata.version(name)
            return True
        except importlib_metadata.PackageNotFoundError:
            return False

    gpu_runtime_installed = distribution_is_installed("onnxruntime-gpu")
    if distribution_is_installed("onnxruntime"):
        # Both wheels own the same import package. Remove the CPU wheel;
        # if the GPU wheel already existed, restore any shared files.
        run(sys.executable, "-m", "pip", "uninstall", "-q", "-y", "onnxruntime")
        if gpu_runtime_installed:
            run(
                sys.executable, "-m", "pip", "install", "-q",
                "--force-reinstall", "--no-deps", "onnxruntime-gpu",
            )
    run(sys.executable, "-m", "pip", "install", "-q", *DEPENDENCIES)

    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
    required_event_files = [
        Path("dataframes/signal.parquet"),
        Path("dataframes/background.parquet"),
    ]
    if REMAKE_EVENTS or any(not path.exists() for path in required_event_files):
        run(
            sys.executable,
            TUTORIAL_DIR / "generate_distributions.py",
            "--n_bkg", N_BKG,
            "--n_sig", N_SIG,
            "--skip-data",
            "--skip-plots",
        )

print("Working dir:", os.getcwd())


## Statistical construction at a glance

Let $\mu$ denote the physical parameter that labels the simulator law, and let $\nu$ denote the coordinate of the frozen hNDE likelihood. Under simulator misspecification, the hNDE MLE need not converge to $\mu$. Its population target is instead

$$
g(\mu)
=\arg\max_{\nu\geq0}
\mathbb E_{\mathcal D\sim p_{\rm sim}(\cdot\mid\mu)}
[\log L_{\rm H}(\nu;\mathcal D)].
$$

We therefore use the response-matched ordering statistic

$$
t_\mu^{(g)}(\mathcal D)
=-2\log\frac{L_{\rm H}(g(\mu);\mathcal D)}
                 {L_{\rm H}(\widehat\nu;\mathcal D)}
\geq0.
$$

The physical Neyman parameter remains $\mu$ everywhere: flow contexts, classifier inputs, cutoff curves, the auditor, and confidence-set inversion all use $\mu$. Only the hNDE generation coordinate and the likelihood numerator use $g(\mu)$.

Response-matched hNDE toys are generated at $\nu=g(\mu)$ and evaluated with the same numerator. A conditional spline $q_\phi$ and a matched classifier learn their density. In $y=\log(t^{(g)}+\epsilon)$,

$$
C_\phi(\mu)=\int_{\log\epsilon}^{\infty}q_\phi(v\mid\mu)\,dv,
\qquad
\widetilde q_\phi(y\mid\mu)
=\frac{q_\phi(y\mid\mu)\,\mathbb I[y\geq\log\epsilon]}
       {C_\phi(\mu)},
$$

and

$$
p_{\rm H}^{(g)}(y\mid\mu)
=\frac{\widetilde q_\phi(y\mid\mu)r_1(y,\mu)}
       {\widetilde Z_1(\mu)},
\qquad
F_{\rm H}^{(g)}(t\mid\mu)
=\int_{\log\epsilon}^{\log(t+\epsilon)}
p_{\rm H}^{(g)}(y\mid\mu)\,dy.
$$

A simulator toy generated at physical truth $\mu$ is evaluated with $t_\mu^{(g)}$ and mapped to

$$
u_0=F_{\rm H}^{(g)}(t_\mu^{(g)}\mid\mu)\in[0,1].
$$

If response matching had removed every simulator discrepancy, $u_0\mid\mu$ would be uniform. The second matched classifier estimates

$$
r_{\rm cal}(u,\mu)
=\frac{p_{\rm sim}^{U_0}(u\mid\mu)}{\mathrm{Uniform}(u)}
=p_{\rm sim}^{U_0}(u\mid\mu).
$$

Finite neural odds are normalized explicitly,

$$
G(u\mid\mu)
=\frac{\int_0^u r_{\rm cal}(v,\mu)\,dv}
       {\int_0^1 r_{\rm cal}(v,\mu)\,dv},
$$

giving

$$
F_{\rm cal}(t\mid\mu)
=G\!\left(F_{\rm H}^{(g)}(t\mid\mu)\mid\mu\right),
\qquad
U_{\rm cal}=F_{\rm cal}(T_\mu^{(g)}\mid\mu).
$$

In the oracle continuous limit, $U_{\rm cal}\mid\mu\sim\mathrm{Uniform}(0,1)$. Because large $t_\mu^{(g)}$ rejects the tested value, the 95% acceptance rule is $U_{\rm cal}\leq0.95$. The PIT convention is the upper-CDF convention appropriate to a likelihood-ratio statistic; it reverses the lower-quantile sign convention used in the [LF2I paper](https://arxiv.org/abs/2107.03920).


In [ ]:
import gc
import hashlib
import importlib
import os
from pathlib import Path

# JAX fits the toy batches before PyTorch trains the large spline. Avoid
# reserving the whole Colab GPU so both frameworks can share it.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import pyarrow.parquet as pq
import sklearn
from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.stats import chi2, norm

import torch

import nsbi_common_utils.training.utils as common_training_utils
import utils as utils_module
import utils_distributions as utils_distributions_module
import utils_hnpe as utils_hnpe_module
import utils_neyman as utils_neyman_module
import utils_nf as utils_nf_module
import utils_plotting as utils_plotting_module
for helper_module in (
    common_training_utils, utils_module, utils_distributions_module,
    utils_hnpe_module, utils_neyman_module, utils_nf_module,
    utils_plotting_module,
):
    importlib.reload(helper_module)

from nsbi_common_utils.training.utils import load_trained_model
from utils import FEATURES, predict_with_model
from utils_distributions import (
    background_components,
    signal_components,
    smearing_parameters,
)
from utils_hnpe import (
    train_ratio_classifier,
    train_spline_flow,
)
from utils_neyman import (
    asimov_test_statistic,
    binned_coverage,
    build_compressed_q_model,
    conditional_cdf_values,
    conditional_density_grid,
    conditional_quantiles,
    conditional_ratio_grid,
    conditional_row_quantiles,
    load_streamed_simulator_template,
    conservative_empirical_quantile,
    coverage_auditor_probability,
    run_cached_toy_ensemble,
    run_streamed_simulator_template,
    sample_truncated_spline_flow,
    simulator_templates_from_exercise5,
    train_coverage_auditor,
    wilson_interval,
)
from utils_nf import (
    accumulate_preselection_histogram,
    checkpoint_path,
    choose_preselection_ratio_cut,
    collect_preselected_eval_rows,
    flow_sample_x,
    load_flow,
)
from utils_plotting import export_standalone_figure_script

FEATURES = list(FEATURES)
SEED = 11082026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Configuration and frozen Exercise 5 inputs

`FAST_MODE=True` is only a structural test. The default run builds two independent simulator streams—construction and sealed audit—with **64 million post-PRESEL events per process**. Each stream also stores exact nested prefixes at 250k, 1M, 4M, and 16M. “64M events” therefore means 64M selected signal events *and* 64M selected background events, not 64M raw proposals or a signal-background total.

Only small integer histograms and provenance are written to Drive. Disk capacity is not the bottleneck: at the prior observed PRESEL acceptances (about 55.6% signal and 15.3% background), the symmetric construction and audit require roughly 1.1 billion raw PRESEL evaluations and 256 million selected rows, corresponding to about two billion member-level ratio-network forwards. The full run is expected to span multiple Colab sessions. Deterministic per-batch seeds, semantic canaries, and atomic summaries permit exact validated resumption after an interruption.

The final calibration uses 500,000 uniform simulator toys plus 250,000 low-$\mu$ toys, a 250,000-toy same-template control, and a 500,000-toy sealed audit. The new audit seeds differ from the already opened 250k audit in the previous run. Its identity does **not** depend on the construction fingerprint, so changing the construction cannot silently recycle the same audit draw under a new filename.

The full stream requires ONNX Runtime's CUDA provider. The notebook prints the actual provider and stops early unless CUDA is active or `ALLOW_SLOW_CPU_TEMPLATE_BUILD=True` is set deliberately. A CUDA-capable PyTorch installation alone does not imply that ONNX inference is on the GPU.

The hNDE event/reference model files remain frozen. A previously certified compressed cache is reused only if it carries a matching SHA-256 fingerprint of those exact files; the legacy unbound cache is rebuilt once rather than being trusted implicitly. Only $g$, the response-matched sampling law, PIT correction, endpoints, cutoffs, auditor, and inversion are statistically changed by the 64M construction.


In [ ]:
BASE_PATH = Path("dataframes")
PRESEL_MODEL_DIR = Path("models_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference_spline16_tail5")
RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}
HYBRID_DENSITY_DIR = Path("saved_densities_hybrid")
EXERCISE5_RATIO_NORMALIZATION_PATH = (
    HYBRID_DENSITY_DIR / "ratio_normalization.npz"
)
if EXERCISE5_RATIO_NORMALIZATION_PATH.exists():
    with np.load(EXERCISE5_RATIO_NORMALIZATION_PATH) as saved_norm:
        EXERCISE5_RATIO_NORMALIZATION = {
            "signal": float(saved_norm["signal"]),
            "background": float(saved_norm["background"]),
        }
    EXERCISE5_NORMALIZATION_SOURCE = str(
        EXERCISE5_RATIO_NORMALIZATION_PATH
    )
else:
    # Legacy Exercise 5 saved normalized ratio arrays but not these two
    # finite-reference means. These values are transcribed from the committed
    # full-run output and are used only to transport the legacy diagnostic.
    EXERCISE5_RATIO_NORMALIZATION = {
        "signal": 0.999149,
        "background": 0.999669,
    }
    EXERCISE5_NORMALIZATION_SOURCE = "committed Exercise 5 output"

FAST_MODE = False
LOAD_IF_AVAILABLE = True
ALLOW_SLOW_CPU_TEMPLATE_BUILD = False
BASE_RUN_TAG = "fast_response_stream_v3" if FAST_MODE else "full_response_stream64m_v3"
PIT_RUN_TAG = "fast_pit_stream_v3" if FAST_MODE else "full_pit_stream64m_v3"
MODEL_DIR = Path("models_exercise11_hybrid_neyman_v1") / BASE_RUN_TAG
CACHE_DIR = Path("saved_exercise11_hybrid_neyman_v1") / BASE_RUN_TAG
PIT_MODEL_DIR = Path("models_exercise11_hybrid_neyman_v1") / PIT_RUN_TAG
PIT_CACHE_DIR = Path("saved_exercise11_hybrid_neyman_v1") / PIT_RUN_TAG
PLOT_DIR = Path("plots_exercise11_hybrid_neyman_v1") / PIT_RUN_TAG
SIMULATOR_TEMPLATE_DIR = Path("saved_exercise11_simulator_stream_v5")
FIGURE_SCRIPT_DIR = Path("exercise11_figures_scripts")
RATIO1_MODEL_DIR = MODEL_DIR / "hnde_residual_ensemble4"
CALIBRATION_RATIO_MODEL_DIR = PIT_MODEL_DIR / "pit_calibration_ensemble4"
for directory in [
    MODEL_DIR, CACHE_DIR, PIT_MODEL_DIR, PIT_CACHE_DIR, PLOT_DIR,
    FIGURE_SCRIPT_DIR, SIMULATOR_TEMPLATE_DIR, RATIO1_MODEL_DIR,
    CALIBRATION_RATIO_MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SAMPLE_PATHS = {
    "signal": BASE_PATH / "signal.parquet",
    "background": BASE_PATH / "background.parquet",
}
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.92
STREAM_BATCH_SIZE = 100_000
PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO = 250.0
PRESEL_CUT_HISTOGRAM_BINS = 4_000
PRESEL_LOG_RATIO_RANGE = (-20.0, 20.0)
REFERENCE_FLOW_TYPE = "quadratic_spline"
REFERENCE_SAMPLING_BATCH_SIZE = 65_536
RATIO_ENSEMBLE_SIZE = 4
PRESEL_EVALUATION_BATCH_SIZE = 100_000
RATIO_EVALUATION_BATCH_SIZE = 100_000
RATIO_FLOOR = 1.0e-12

MU_RANGE = (0.0, 3.0)
LOW_MU_CALIBRATION_RANGE = (0.0, 0.75)
TEMPLATE_SELECTED_CHECKPOINTS = np.asarray(
    [10_000, 50_000] if FAST_MODE
    else [250_000, 1_000_000, 4_000_000, 16_000_000, 64_000_000],
    dtype=np.int64,
)
FINAL_TEMPLATE_EVENTS = int(TEMPLATE_SELECTED_CHECKPOINTS[-1])
FRESH_SIMULATOR_BATCH_SIZE = 100_000 if FAST_MODE else 500_000
TEMPLATE_SAVE_EVERY_BATCHES = 2 if FAST_MODE else 10
TEMPLATE_PROGRESS_EVERY_BATCHES = 2 if FAST_MODE else 10
FEATURE_DIAGNOSTIC_BINS = 64 if FAST_MODE else 128
FRESH_CONSTRUCTION_SEED = {
    "signal": SEED + 2_601,
    "background": SEED + 2_602,
}
# The +1601/+1602 audit was opened in the preceding run. These +3601/+3602
# seeds define a genuinely new audit that has not influenced this method.
FRESH_AUDIT_SEED = {
    "signal": SEED + 3_601,
    "background": SEED + 3_602,
}
CONSTRUCTION_TEMPLATE_PATHS = {
    process: SIMULATOR_TEMPLATE_DIR
    / f"construction_{process}_selected_{FINAL_TEMPLATE_EVENTS}.npz"
    for process in ("signal", "background")
}
SEALED_AUDIT_TEMPLATE_PATHS = {
    process: SIMULATOR_TEMPLATE_DIR
    / f"sealed_audit_{process}_selected_{FINAL_TEMPLATE_EVENTS}.npz"
    for process in ("signal", "background")
}
EXERCISE5_CLOSURE_PATHS = {
    process: SIMULATOR_TEMPLATE_DIR / f"exercise5_runtime_closure_{process}.npz"
    for process in ("signal", "background")
}

TOY_Q_BINS = 512
TOY_BATCH_SIZE = 2_000 if FAST_MODE else 10_000
TOY_NEWTON_STEPS = 16
TOY_MU_MAX = 12.0
TOY_FIT_FINGERPRINT = "jax_newton16_bisection64_mumax12_v2"
TOY_FIT_RUNTIME_FINGERPRINT = "pending_toy_runtime_canary"
T_OFFSET = 1.0e-6
LOG_RATIO_CLIP = 15.0
PIT_EPS = 1.0e-6
QUANTILE_LEVELS = np.asarray([0.50, 0.68, 0.90, 0.95, 0.99])
ANCHOR_MUS = np.asarray([0.0, 3.0])
PREFIX_TRANSFER_MUS = np.asarray([0.0, 0.25, 0.50, 1.0, 2.0, 3.0])

if FAST_MODE:
    N_REFERENCE_EVENTS = 250_000
    N_HNDE_TOYS = 50_000
    N_FLOW_TOYS = 40_000
    N_RATIO1_TOYS = 10_000
    N_SIMULATOR_CALIBRATION_TOYS = 20_000
    N_SIMULATOR_LOW_MU_TOYS = 10_000
    N_SIMULATOR_INTERNAL_AUDIT_TOYS = 20_000
    N_SIMULATOR_AUDIT_TOYS = 20_000
    N_TEMPLATE_BOOTSTRAPS = 4
    N_TOYS_PER_TEMPLATE_BOOTSTRAP = 2_000
    N_PREFIX_TRANSFER_TOYS = 2_000
    RESPONSE_GRID_POINTS = 301
    N_ANCHOR_TOYS = 5_000
    N_AUDIT_ANCHOR_TOYS = 5_000
    FLOW_EPOCHS = 8
    RATIO_EPOCHS = 12
    QUADRATURE_MU_POINTS = 101
    QUADRATURE_Y_POINTS = 1_025
    PIT_QUADRATURE_POINTS = 1_025
else:
    N_REFERENCE_EVENTS = 5_000_000
    N_HNDE_TOYS = 500_000
    N_FLOW_TOYS = 400_000
    N_RATIO1_TOYS = 100_000
    N_SIMULATOR_CALIBRATION_TOYS = 500_000
    N_SIMULATOR_LOW_MU_TOYS = 250_000
    N_SIMULATOR_INTERNAL_AUDIT_TOYS = 250_000
    N_SIMULATOR_AUDIT_TOYS = 500_000
    N_TEMPLATE_BOOTSTRAPS = 20
    N_TOYS_PER_TEMPLATE_BOOTSTRAP = 20_000
    N_PREFIX_TRANSFER_TOYS = 50_000
    RESPONSE_GRID_POINTS = 1_001
    N_ANCHOR_TOYS = 50_000
    N_AUDIT_ANCHOR_TOYS = 50_000
    FLOW_EPOCHS = 70
    RATIO_EPOCHS = 50
    QUADRATURE_MU_POINTS = 301
    QUADRATURE_Y_POINTS = 4_097
    PIT_QUADRATURE_POINTS = 4_097

if N_FLOW_TOYS + N_RATIO1_TOYS != N_HNDE_TOYS:
    raise ValueError("The disjoint hNDE flow/ratio splits must exhaust the toys.")

STATISTIC_FLOW_MODEL_CONFIG = {
    "n_coupling_layers": 10,
    "hidden_features": 1024,
    "hidden_layers": 4,
    "spline_num_bins": 16,
    "spline_tail_bound": 5.0,
    "dropout_probability": 0.0,
}
STATISTIC_FLOW_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": FLOW_EPOCHS,
    "learning_rate": 1.0e-4,
    "lr_scheduler_factor": 0.2,
    "lr_scheduler_patience": 2,
    "min_learning_rate": 1.0e-7,
    "weight_decay": 0.0,
    "validation_fraction": 0.20,
    "patience": 5,
    "gradient_clip": 5.0,
}
CORRECTION_MODEL_CONFIG = {
    "hidden_features": 1024,
    "hidden_layers": 4,
    "dropout_probability": 0.0,
}
CORRECTION_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": RATIO_EPOCHS,
    "learning_rate": 1.0e-3,
    "lr_scheduler": "step",
    "lr_scheduler_factor": 0.01,
    "lr_scheduler_patience": 10,
    "validation_fraction": 0.20,
    "patience": 20,
    "gradient_clip": 5.0,
}
AUDITOR_MODEL_CONFIG = {"hidden_features": 64, "hidden_layers": 3}
AUDITOR_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": 200 if not FAST_MODE else 40,
    "learning_rate": 1.0e-3,
    "weight_decay": 1.0e-4,
    "lr_scheduler_factor": 0.3,
    "lr_scheduler_patience": 5,
    "min_learning_rate": 1.0e-6,
    "validation_fraction": 0.20,
    "patience": 20,
    "gradient_clip": 5.0,
}


def export_exercise11_figure(fig, script_name):
    fig.savefig(PLOT_DIR / f"{script_name}.png", dpi=160)
    return export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )


required_paths = [
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
    checkpoint_path("reference", REFERENCE_FLOW_MODEL_DIR, REFERENCE_FLOW_TYPE),
    HYBRID_DENSITY_DIR / "weights_asimov.npy",
    HYBRID_DENSITY_DIR / "ratio_signal_asimov.npy",
    HYBRID_DENSITY_DIR / "ratio_background_asimov.npy",
]
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    for member in range(RATIO_ENSEMBLE_SIZE):
        required_paths.extend([
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        ])
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Exercise 11 loads the frozen Exercise 5 model and held-out arrays. "
        "Run Exercise 5 through 'Reconstruct and validate the hybrid densities' "
        "first. Missing:\n" + "\n".join(f"  - {path}" for path in missing_paths)
    )

print(f"Base/PIT run tags: {BASE_RUN_TAG} / {PIT_RUN_TAG}")
print("Nested selected-event checkpoints per process:", TEMPLATE_SELECTED_CHECKPOINTS)
print(f"hNDE toys: {N_HNDE_TOYS:,}")
print(
    "simulator uniform/low-mu calibration toys: "
    f"{N_SIMULATOR_CALIBRATION_TOYS:,}/{N_SIMULATOR_LOW_MU_TOYS:,}"
)
print(
    "internal/sealed audit toys: "
    f"{N_SIMULATOR_INTERNAL_AUDIT_TOYS:,}/{N_SIMULATOR_AUDIT_TOYS:,}"
)
print(f"Standalone figure scripts: {FIGURE_SCRIPT_DIR}/")


## Load the frozen hNDE event model

The PRESEL classifier, post-selection yields, reference flow, and the two four-member density-ratio ensembles are the same objects used by Exercise 5. No event-level model is retrained here.


In [ ]:
def as_inference_session(model_candidate):
    if isinstance(model_candidate, ort.InferenceSession):
        return model_candidate
    available = ort.get_available_providers()
    providers = [
        provider
        for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if provider in available
    ] or available
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    return ort.InferenceSession(
        model_candidate.SerializeToString(),
        sess_options=options,
        providers=providers,
    )


PRESEL_scaler, PRESEL_model_proto = load_trained_model(
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
)
PRESEL_model = as_inference_session(PRESEL_model_proto)
del PRESEL_model_proto


def evaluate_PRESEL_ratio(features):
    if isinstance(features, pd.DataFrame):
        feature_dataframe = features.loc[:, FEATURES].astype(
            "float32", copy=False
        )
    else:
        values = np.asarray(features, dtype=np.float32)
        if values.ndim != 2 or values.shape[1] != len(FEATURES):
            raise ValueError(
                "PRESEL features must have shape (n_events, n_features)."
            )
        # Exercise 5's ColumnTransformer selects columns by name.
        feature_dataframe = pd.DataFrame(values, columns=FEATURES)
    ratio = predict_with_model(
        feature_dataframe,
        scaler=PRESEL_scaler,
        model=PRESEL_model,
        batch_size=PRESEL_EVALUATION_BATCH_SIZE,
    )
    return np.asarray(ratio, dtype=np.float64).reshape(-1)


PRESEL_STATE_CANDIDATES = [
    CACHE_DIR / "exercise5_preselection_state.npz",
    Path("saved_asimov_nis_influence_v2/exercise5_preselection_state.npz"),
    Path("saved_exercise7_misspecification/exercise5_preselection_state.npz"),
]
existing_state = next(
    (path for path in PRESEL_STATE_CANDIDATES if path.exists()), None
)
if existing_state is not None:
    state = np.load(existing_state)
    PRESEL_RATIO_CUT = float(state["ratio_cut"])
    LAM_SIG = float(state["lambda_signal"])
    LAM_BKG = float(state["lambda_background"])
    print(f"Loaded PRESEL state from {existing_state}")
else:
    edges = np.linspace(
        PRESEL_LOG_RATIO_RANGE[0], PRESEL_LOG_RATIO_RANGE[1],
        PRESEL_CUT_HISTOGRAM_BINS + 1,
    )
    histograms, statistics = {}, {}
    for sample_name in ["signal", "background"]:
        histograms[sample_name], statistics[sample_name] = (
            accumulate_preselection_histogram(
                SAMPLE_PATHS[sample_name],
                features=FEATURES,
                ratio_predictor=evaluate_PRESEL_ratio,
                log_ratio_edges=edges,
                batch_size=STREAM_BATCH_SIZE,
                presel_fraction=PRESEL_TRAIN_FRACTION,
                flow_train_fraction=FLOW_TRAIN_FRACTION,
                split_seed=SPLIT_SEED,
            )
        )
    PRESEL_RATIO_CUT, diagnostics = choose_preselection_ratio_cut(
        histograms["signal"], histograms["background"], edges,
        signal_inclusive_yield=statistics["signal"]["inclusive_weight"],
        background_inclusive_yield=statistics["background"]["inclusive_weight"],
        signal_partition_weight=statistics["signal"]["partition_weight"],
        background_partition_weight=statistics["background"]["partition_weight"],
        target_background_to_signal=PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO,
    )
    LAM_SIG = diagnostics["histogram_signal_yield"]
    LAM_BKG = diagnostics["histogram_background_yield"]
    np.savez(
        PRESEL_STATE_CANDIDATES[0],
        ratio_cut=PRESEL_RATIO_CUT,
        lambda_signal=LAM_SIG,
        lambda_background=LAM_BKG,
    )

reference_flow = load_flow(
    "reference",
    model_dir=REFERENCE_FLOW_MODEL_DIR,
    flow_type=REFERENCE_FLOW_TYPE,
    device=device,
    expected_features=FEATURES,
)
ratio_models = {}
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    ratio_models[sample_name] = []
    for member in range(RATIO_ENSEMBLE_SIZE):
        scaler, model_proto = load_trained_model(
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        )
        ratio_models[sample_name].append(
            {"scaler": scaler, "model": as_inference_session(model_proto)}
        )

event_sessions = [PRESEL_model] + [
    pack["model"]
    for process_packs in ratio_models.values()
    for pack in process_packs
]
ACTIVE_EVENT_PROVIDERS = [session.get_providers()[0] for session in event_sessions]
print("ONNX Runtime version/providers:", ort.__version__, sorted(set(ACTIVE_EVENT_PROVIDERS)))
if (
    not FAST_MODE
    and not ALLOW_SLOW_CPU_TEMPLATE_BUILD
    and any(provider != "CUDAExecutionProvider" for provider in ACTIVE_EVENT_PROVIDERS)
):
    raise RuntimeError(
        "The 64M construction requires CUDAExecutionProvider for every event "
        "network. Colab should install onnxruntime-gpu in the setup cell. "
        "Restart the runtime after installation, or set "
        "ALLOW_SLOW_CPU_TEMPLATE_BUILD=True only if a day-scale CPU run is "
        "intentional."
    )

print(f"PRESEL ratio cut: {PRESEL_RATIO_CUT:.6g}")
print(f"Post-selection yields: signal={LAM_SIG:.6g}, background={LAM_BKG:.6g}")


## Reconstruct the frozen hNDE likelihood and stream the simulator laws

The hNDE likelihood compression remains the one used in the preceding run. Exercise 5's saved 250k-per-process arrays are retained only as a labeled legacy closure point; they no longer define the construction.

Two fresh analytic-simulator streams are then generated. The reconstructed Gaussian mixture is sampled directly after analytically convolving the latent covariance with the detector smearing, which is exactly distribution-equivalent to drawing latent $z$ and then drawing $x\mid z$. Every raw batch is passed through the frozen PRESEL network. Passing rows are immediately evaluated by the two four-member ratio ensembles and reduced to:

- a 512-bin histogram of $\log q(x)$;
- one fixed histogram for each reconstructed feature;
- generated, passing, and selected counters.

Exact nested snapshots are taken at 250k, 1M, 4M, 16M, and 64M selected events. A deterministic seed is derived from `(stream seed, raw batch index)`, so resuming from the last atomic state reproduces the uninterrupted result bit for bit. Before resumption, a 100,003-event-per-process end-to-end semantic canary fingerprints PRESEL decisions and the selected $q$/feature bin assignments while exercising one full 100k inference batch plus a remainder. A partial cache requires the same runtime fingerprint; a complete immutable cache may be loaded elsewhere, but construction and audit caches must carry the same recorded inference fingerprint before they are combined. Only the construction counts are exposed here. The independently seeded audit returns provenance and scalar throughput only; its $q$ and feature laws remain unopened until the final audit section.


In [ ]:
def evaluate_ratio(sample_name, values, batch_size=RATIO_EVALUATION_BATCH_SIZE):
    values = np.asarray(values, dtype=np.float32)
    chunks = []
    for start in range(0, len(values), int(batch_size)):
        batch = pd.DataFrame(
            values[start : start + int(batch_size)], columns=FEATURES
        )
        member_predictions = []
        for pack in ratio_models[sample_name]:
            prediction = predict_with_model(
                batch, scaler=pack["scaler"], model=pack["model"],
                batch_size=int(batch_size),
            )
            member_predictions.append(
                np.asarray(prediction, dtype=np.float64).reshape(-1)
            )
        chunks.append(np.mean(np.stack(member_predictions, axis=0), axis=0))
    ratio = np.concatenate(chunks) if chunks else np.empty(0)
    if not np.isfinite(ratio).all():
        raise FloatingPointError(f"Non-finite {sample_name} ratio.")
    return np.maximum(ratio, RATIO_FLOOR)


def sample_preselected_flow(flow_pack, n_events, batch_size=65_536):
    accepted_chunks = []
    n_kept = 0
    n_generated = 0
    n_passed = 0
    while n_kept < int(n_events):
        needed = int(n_events) - n_kept
        current_batch = max(int(batch_size), min(4 * int(batch_size), 2 * needed))
        generated = flow_sample_x(flow_pack, current_batch, batch_size=batch_size)
        generated_df = pd.DataFrame(generated, columns=FEATURES)
        passes = evaluate_PRESEL_ratio(generated_df) >= PRESEL_RATIO_CUT
        n_generated += len(generated)
        n_passed += int(passes.sum())
        if np.any(passes):
            accepted_chunks.append(generated[passes])
            n_kept += int(passes.sum())
    accepted = np.concatenate(accepted_chunks, axis=0)[: int(n_events)]
    return accepted.astype(np.float32, copy=False), n_passed / n_generated


def sha256_file(path, chunk_size=2**20):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(int(chunk_size))
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


FROZEN_EVENT_MODEL_PATHS = [
    Path(reference_flow["path"]),
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
]
for model_dir in RATIO_MODEL_DIR.values():
    for member in range(RATIO_ENSEMBLE_SIZE):
        FROZEN_EVENT_MODEL_PATHS.extend([
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        ])
frozen_model_digest = hashlib.sha256()
for path in sorted(FROZEN_EVENT_MODEL_PATHS, key=lambda value: str(value)):
    frozen_model_digest.update(str(path).encode("utf-8"))
    frozen_model_digest.update(sha256_file(path).encode("ascii"))
FROZEN_EVENT_MODEL_FINGERPRINT = frozen_model_digest.hexdigest()
print("Frozen event/reference model fingerprint:", FROZEN_EVENT_MODEL_FINGERPRINT)


CERTIFIED_COMPRESSED_MODEL_PATH = Path(
    "saved_exercise11_hybrid_neyman_v1/full_response_v2/"
    "compressed_exercise5_model.npz"
)
COMPRESSED_MODEL_PATH = CACHE_DIR / "compressed_exercise5_model.npz"
if not FAST_MODE and CERTIFIED_COMPRESSED_MODEL_PATH.exists():
    with np.load(CERTIFIED_COMPRESSED_MODEL_PATH, allow_pickle=False) as certified:
        certified_fingerprint = (
            str(certified["event_model_fingerprint"].item())
            if "event_model_fingerprint" in certified.files else None
        )
    if certified_fingerprint == FROZEN_EVENT_MODEL_FINGERPRINT:
        COMPRESSED_MODEL_PATH = CERTIFIED_COMPRESSED_MODEL_PATH
    else:
        print(
            "Refusing the legacy compressed cache because it is not "
            "bound to the current event/reference model files; rebuilding."
        )
compression_cache_metadata = {
    "cache_version": 2,
    "n_reference_events": N_REFERENCE_EVENTS,
    "toy_q_bins": TOY_Q_BINS,
    "lambda_signal": LAM_SIG,
    "lambda_background": LAM_BKG,
    "presel_ratio_cut": PRESEL_RATIO_CUT,
    "event_model_fingerprint": FROZEN_EVENT_MODEL_FINGERPRINT,
}
if COMPRESSED_MODEL_PATH.exists():
    saved = np.load(COMPRESSED_MODEL_PATH)
    missing_metadata = set(compression_cache_metadata) - set(saved.files)
    mismatched_metadata = []
    for name, expected in compression_cache_metadata.items():
        if name not in saved.files:
            continue
        observed = np.asarray(saved[name]).item()
        matches = (
            str(observed) == str(expected)
            if isinstance(expected, str)
            else np.isclose(float(observed), float(expected), rtol=1e-12)
        )
        if not matches:
            mismatched_metadata.append(name)
    if missing_metadata or mismatched_metadata:
        raise RuntimeError(
            "The compressed-model cache has stale provenance. Bump "
            "BASE_RUN_TAG (recommended) or remove only that versioned cache. "
            f"Missing={sorted(missing_metadata)}, "
            f"mismatched={mismatched_metadata}."
        )
    COMPRESSED_Q = saved["q"]
    HNDE_SIGNAL_PROBABILITY = saved["signal_probability"]
    HNDE_BACKGROUND_PROBABILITY = saved["background_probability"]
    LOG_Q_EDGES = saved["log_q_edges"]
    RATIO_NORMALIZATION = {
        "signal": float(saved["signal_normalization"]),
        "background": float(saved["background_normalization"]),
    }
    compression_truth = saved["validation_truth"]
    compression_test = saved["validation_test"]
    compression_unbinned = saved["validation_unbinned"]
    compression_binned = saved["validation_binned"]
    saved.close()
    print(f"Loaded compressed hNDE model from {COMPRESSED_MODEL_PATH}")
else:
    torch.manual_seed(SEED + 100)
    reference_values, reference_acceptance = sample_preselected_flow(
        reference_flow, N_REFERENCE_EVENTS, REFERENCE_SAMPLING_BATCH_SIZE
    )
    raw_signal = evaluate_ratio("signal", reference_values)
    raw_background = evaluate_ratio("background", reference_values)
    RATIO_NORMALIZATION = {
        "signal": float(raw_signal.mean()),
        "background": float(raw_background.mean()),
    }
    ratio_signal = raw_signal / RATIO_NORMALIZATION["signal"]
    ratio_background = raw_background / RATIO_NORMALIZATION["background"]
    weight_signal = ratio_signal / N_REFERENCE_EVENTS
    weight_background = ratio_background / N_REFERENCE_EVENTS
    event_q = (
        LAM_SIG / LAM_BKG * ratio_signal / ratio_background
    )
    compressed = build_compressed_q_model(
        event_q,
        weight_signal,
        weight_background,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        n_bins=TOY_Q_BINS,
    )
    COMPRESSED_Q = compressed["q"]
    HNDE_SIGNAL_PROBABILITY = compressed["signal_probability"]
    HNDE_BACKGROUND_PROBABILITY = compressed["background_probability"]
    LOG_Q_EDGES = compressed["log_q_edges"]

    compression_rows = []
    for truth_mu in [0.0, 0.25, 1.0, 2.0, 3.0]:
        unbinned_expected = (
            truth_mu * LAM_SIG * weight_signal
            + LAM_BKG * weight_background
        )
        binned_expected = (
            truth_mu * LAM_SIG * HNDE_SIGNAL_PROBABILITY
            + LAM_BKG * HNDE_BACKGROUND_PROBABILITY
        )
        for test_mu in [0.0, 0.5, 1.0, 2.0, 3.0]:
            if test_mu == truth_mu:
                continue
            compression_rows.append((
                truth_mu,
                test_mu,
                asimov_test_statistic(
                    test_mu, truth_mu, event_q, unbinned_expected,
                    lam_signal=LAM_SIG,
                ),
                asimov_test_statistic(
                    test_mu, truth_mu, COMPRESSED_Q, binned_expected,
                    lam_signal=LAM_SIG,
                ),
            ))
    compression_rows = np.asarray(compression_rows, dtype=np.float64)
    compression_truth = compression_rows[:, 0]
    compression_test = compression_rows[:, 1]
    compression_unbinned = compression_rows[:, 2]
    compression_binned = compression_rows[:, 3]
    np.savez_compressed(
        COMPRESSED_MODEL_PATH,
        q=COMPRESSED_Q,
        signal_probability=HNDE_SIGNAL_PROBABILITY,
        background_probability=HNDE_BACKGROUND_PROBABILITY,
        log_q_edges=LOG_Q_EDGES,
        signal_normalization=RATIO_NORMALIZATION["signal"],
        background_normalization=RATIO_NORMALIZATION["background"],
        validation_truth=compression_truth,
        validation_test=compression_test,
        validation_unbinned=compression_unbinned,
        validation_binned=compression_binned,
        **compression_cache_metadata,
    )
    print(f"Saved compressed hNDE model to {COMPRESSED_MODEL_PATH}")
    print(f"Reference PRESEL acceptance: {reference_acceptance:.3%}")
    del reference_values, raw_signal, raw_background
    del ratio_signal, ratio_background, weight_signal, weight_background, event_q
    gc.collect()

print("Ratio normalizations:", RATIO_NORMALIZATION)
print("Compressed q quantiles:", np.quantile(COMPRESSED_Q, [0, .01, .5, .99, 1]))
# The reference flow is no longer needed once the compressed likelihood is
# loaded or rebuilt. Release its CUDA allocation before nine ORT sessions
# score the construction and audit streams.
reference_flow = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --------------------------------------------------------------------------
# Keep Exercise 5's 250k law only as a legacy diagnostic. The construction
# and unopened audit below are new, independent analytic-simulator streams.
# --------------------------------------------------------------------------
legacy_simulator_templates = simulator_templates_from_exercise5(
    weights_path=HYBRID_DENSITY_DIR / "weights_asimov.npy",
    ratio_signal_path=HYBRID_DENSITY_DIR / "ratio_signal_asimov.npy",
    ratio_background_path=HYBRID_DENSITY_DIR / "ratio_background_asimov.npy",
    log_q_edges=LOG_Q_EDGES,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    seed=SEED + 600,
    source_ratio_normalization=EXERCISE5_RATIO_NORMALIZATION,
    target_ratio_normalization=RATIO_NORMALIZATION,
)
LEGACY_EXERCISE5_SIGNAL_COUNTS = np.asarray(
    legacy_simulator_templates["pooled_signal_counts"], dtype=np.int64
)
LEGACY_EXERCISE5_BACKGROUND_COUNTS = np.asarray(
    legacy_simulator_templates["pooled_background_counts"], dtype=np.int64
)
LEGACY_EXERCISE5_SIGNAL_PROBABILITY = np.asarray(
    legacy_simulator_templates["pooled_signal_probability"], dtype=np.float64
)
LEGACY_EXERCISE5_BACKGROUND_PROBABILITY = np.asarray(
    legacy_simulator_templates["pooled_background_probability"], dtype=np.float64
)


def update_array_digest(digest, label, values):
    values = np.ascontiguousarray(np.asarray(values))
    digest.update(str(label).encode("utf-8"))
    digest.update(str(values.dtype).encode("utf-8"))
    digest.update(str(values.shape).encode("utf-8"))
    digest.update(values.view(np.uint8))


def reconstructed_components(components):
    """Analytically convolve each latent Gaussian with detector smearing."""
    scale, resolution = [
        np.asarray(values, dtype=np.float64)
        for values in smearing_parameters()
    ]
    covariance_scale = np.outer(scale, scale)
    resolution_covariance = np.diag(resolution**2)
    return [
        (
            float(fraction),
            scale * np.asarray(mean, dtype=np.float64),
            covariance_scale * np.asarray(covariance, dtype=np.float64)
            + resolution_covariance,
        )
        for fraction, mean, covariance in components
    ]


RECO_COMPONENTS = {
    "signal": reconstructed_components(signal_components()),
    "background": reconstructed_components(background_components()),
}


def sample_nominal_simulator_features(components, n_events, rng):
    """Draw reconstructed features from the exact convolved Gaussian mixture."""
    fractions = np.asarray([component[0] for component in components], dtype=np.float64)
    fractions /= fractions.sum()
    component_counts = rng.multinomial(int(n_events), fractions)
    values = np.empty((int(n_events), len(FEATURES)), dtype=np.float64)
    offset = 0
    for (_, mean, covariance), count in zip(components, component_counts):
        if count:
            values[offset : offset + count] = rng.multivariate_normal(
                mean, covariance, size=int(count)
            )
            offset += int(count)
    rng.shuffle(values, axis=0)
    return values.astype(np.float32, copy=False)


def evaluate_log_q(values):
    values = np.asarray(values, dtype=np.float32)
    ratio_signal = evaluate_ratio("signal", values) / RATIO_NORMALIZATION["signal"]
    ratio_background = (
        evaluate_ratio("background", values) / RATIO_NORMALIZATION["background"]
    )
    return (
        np.log(LAM_SIG / LAM_BKG)
        + np.log(ratio_signal)
        - np.log(ratio_background)
    )


# Fixed, process-independent feature edges derived from the analytic support.
# Infinite overflow bins make loss of selected events a structural error.
all_reco_components = RECO_COMPONENTS["signal"] + RECO_COMPONENTS["background"]
feature_edge_rows = []
for feature_index in range(len(FEATURES)):
    means = np.asarray([
        component[1][feature_index] for component in all_reco_components
    ])
    standard_deviations = np.sqrt(np.asarray([
        component[2][feature_index, feature_index]
        for component in all_reco_components
    ]))
    lower = float(np.min(means - 8.0 * standard_deviations))
    upper = float(np.max(means + 8.0 * standard_deviations))
    interior = np.linspace(lower, upper, FEATURE_DIAGNOSTIC_BINS - 1)
    feature_edge_rows.append(np.concatenate(([-np.inf], interior, [np.inf])))
FEATURE_DIAGNOSTIC_EDGES = np.asarray(feature_edge_rows, dtype=np.float64)


# Hash the actual frozen model files, generator arrays, numerical normalizers,
# ONNX version/provider, and inference batch sizes. A partial cache can never
# mix different event-network numerics silently.
stream_recipe_digest = hashlib.sha256()
stream_recipe_digest.update(b"analytic_reco_gaussian_mixture_stream_v3")
for value in (
    PRESEL_RATIO_CUT,
    RATIO_NORMALIZATION["signal"],
    RATIO_NORMALIZATION["background"],
    LAM_SIG,
    LAM_BKG,
    RATIO_FLOOR,
    PRESEL_EVALUATION_BATCH_SIZE,
    RATIO_EVALUATION_BATCH_SIZE,
):
    update_array_digest(stream_recipe_digest, "numeric", np.asarray(value))
for process in ("signal", "background"):
    for fraction, mean, covariance in RECO_COMPONENTS[process]:
        update_array_digest(stream_recipe_digest, f"{process}_fraction", fraction)
        update_array_digest(stream_recipe_digest, f"{process}_mean", mean)
        update_array_digest(stream_recipe_digest, f"{process}_covariance", covariance)
for path in sorted(FROZEN_EVENT_MODEL_PATHS, key=lambda value: str(value)):
    stream_recipe_digest.update(str(path).encode("utf-8"))
    stream_recipe_digest.update(sha256_file(path).encode("ascii"))

# A deterministic end-to-end semantic canary rejects partial-cache
# resumption exactly when runtime changes alter what the stream retains:
# PRESEL decisions or q/feature bin assignments. Irrelevant last-bit
# differences within the same bins may resume across GPU types.
STREAM_CANARY_EVENTS = 100_003
stream_canary_digest = hashlib.sha256()
for process_index, process in enumerate(("signal", "background")):
    canary_rng = np.random.default_rng(
        np.random.SeedSequence([SEED, 7_771, process_index])
    )
    canary_values = sample_nominal_simulator_features(
        RECO_COMPONENTS[process], STREAM_CANARY_EVENTS, canary_rng
    )
    canary_presel = evaluate_PRESEL_ratio(canary_values)
    canary_passes = (
        np.isfinite(canary_presel)
        & (canary_presel >= PRESEL_RATIO_CUT)
    )
    if not np.any(canary_passes):
        raise RuntimeError(f"The {process} stream canary selected no events.")
    # Score all rows so every ensemble member sees one full 100k
    # inference batch plus a non-divisible remainder.
    canary_log_q = evaluate_log_q(canary_values)
    update_array_digest(stream_canary_digest, f"{process}_passes", canary_passes)
    selected_canary_values = canary_values[canary_passes]
    selected_canary_log_q = canary_log_q[canary_passes]
    canary_q_bins = np.searchsorted(
        LOG_Q_EDGES, selected_canary_log_q, side="right"
    ) - 1
    update_array_digest(stream_canary_digest, f"{process}_q_bins", canary_q_bins)
    for feature_index, feature in enumerate(FEATURES):
        canary_feature_bins = np.searchsorted(
            FEATURE_DIAGNOSTIC_EDGES[feature_index],
            selected_canary_values[:, feature_index], side="right",
        ) - 1
        update_array_digest(
            stream_canary_digest, f"{process}_{feature}_bins",
            canary_feature_bins,
        )
    del canary_values, canary_presel, canary_passes, canary_log_q
    del selected_canary_values, selected_canary_log_q, canary_q_bins
    del canary_feature_bins
STREAM_CANARY_FINGERPRINT = stream_canary_digest.hexdigest()
stream_runtime_digest = hashlib.sha256()
for runtime_value in (
    ort.__version__, np.__version__, pd.__version__, sklearn.__version__,
    "|".join(ACTIVE_EVENT_PROVIDERS), STREAM_CANARY_FINGERPRINT,
):
    stream_runtime_digest.update(str(runtime_value).encode("utf-8"))
STREAM_RUNTIME_FINGERPRINT = (
    "event_inference_runtime_v1_" + stream_runtime_digest.hexdigest()[:24]
)
STREAM_RECIPE_FINGERPRINT = (
    "analytic_reco_stream_v3_" + stream_recipe_digest.hexdigest()[:24]
)
print("End-to-end stream canary:", STREAM_CANARY_FINGERPRINT)
print("Stream inference/runtime fingerprint:", STREAM_RUNTIME_FINGERPRINT)
print("Stream recipe fingerprint:", STREAM_RECIPE_FINGERPRINT)


def exercise5_runtime_closure(process):
    """Summarize the current-runtime Exercise 5 eval split for closure only."""
    path = EXERCISE5_CLOSURE_PATHS[process]
    source_path = SAMPLE_PATHS[process]
    source_stat = source_path.stat()
    source_rows = int(pq.ParquetFile(source_path).metadata.num_rows)
    closure_metadata = {
        "schema": "exercise5_eval_runtime_closure_v3",
        "process": process,
        "stream_recipe": STREAM_RECIPE_FINGERPRINT,
        "stream_runtime": STREAM_RUNTIME_FINGERPRINT,
        "source_path": str(source_path),
        "source_size": int(source_stat.st_size),
        "source_mtime_ns": int(source_stat.st_mtime_ns),
        "source_rows": source_rows,
        "split_seed": int(SPLIT_SEED),
        "presel_train_fraction": float(PRESEL_TRAIN_FRACTION),
        "flow_train_fraction": float(FLOW_TRAIN_FRACTION),
        "stream_batch_size": int(STREAM_BATCH_SIZE),
    }
    recipe = json.dumps(
        closure_metadata, sort_keys=True, separators=(",", ":")
    )
    required = {
        "recipe", "log_q_edges", "feature_edges", "q_counts",
        "feature_counts", "selected_events", "partition_events",
        "source_size", "source_mtime_ns", "source_rows",
    }
    if path.exists():
        try:
            with np.load(path, allow_pickle=False) as saved:
                valid = (
                    required.issubset(saved.files)
                    and str(saved["recipe"].item()) == recipe
                    and np.array_equal(saved["log_q_edges"], LOG_Q_EDGES)
                    and np.array_equal(saved["feature_edges"], FEATURE_DIAGNOSTIC_EDGES)
                )
                if valid:
                    payload = {name: np.asarray(saved[name]) for name in required}
                else:
                    payload = None
        except (OSError, ValueError, KeyError):
            payload = None
    else:
        payload = None
    if payload is None:
        print(f"Streaming Exercise 5 eval rows for generator closure: {process}")
        selected, stats = collect_preselected_eval_rows(
            SAMPLE_PATHS[process],
            features=FEATURES,
            ratio_predictor=evaluate_PRESEL_ratio,
            ratio_cut=PRESEL_RATIO_CUT,
            batch_size=STREAM_BATCH_SIZE,
            presel_fraction=PRESEL_TRAIN_FRACTION,
            flow_train_fraction=FLOW_TRAIN_FRACTION,
            split_seed=SPLIT_SEED,
        )
        values = selected[FEATURES].to_numpy(dtype=np.float32)
        q_counts = np.histogram(evaluate_log_q(values), bins=LOG_Q_EDGES)[0].astype(np.int64)
        feature_counts = np.stack([
            np.histogram(values[:, index], bins=FEATURE_DIAGNOSTIC_EDGES[index])[0]
            for index in range(len(FEATURES))
        ]).astype(np.int64)
        if q_counts.sum() != len(values) or not np.all(feature_counts.sum(axis=1) == len(values)):
            raise RuntimeError("The Exercise 5 closure histograms lost selected events.")
        payload = {
            "recipe": np.asarray(recipe),
            "log_q_edges": LOG_Q_EDGES,
            "feature_edges": FEATURE_DIAGNOSTIC_EDGES,
            "q_counts": q_counts,
            "feature_counts": feature_counts,
            "selected_events": np.asarray(len(values), dtype=np.int64),
            "partition_events": np.asarray(stats["partition_events"], dtype=np.int64),
            "source_size": np.asarray(source_stat.st_size, dtype=np.int64),
            "source_mtime_ns": np.asarray(source_stat.st_mtime_ns, dtype=np.int64),
            "source_rows": np.asarray(source_rows, dtype=np.int64),
        }
        temporary = path.with_suffix(".tmp.npz")
        np.savez_compressed(temporary, **payload)
        temporary.replace(path)
        del selected, values
        gc.collect()
    return {
        "q_counts": np.asarray(payload["q_counts"], dtype=np.int64),
        "feature_counts": np.asarray(payload["feature_counts"], dtype=np.int64),
        "selected_events": int(np.asarray(payload["selected_events"]).item()),
        "partition_events": int(np.asarray(payload["partition_events"]).item()),
    }


LEGACY_RUNTIME_CLOSURE = {
    process: exercise5_runtime_closure(process)
    for process in ("signal", "background")
}
print(
    "Exercise 5 current-runtime PRESEL acceptance reference:",
    {
        process: (summary["selected_events"] / summary["partition_events"])
        for process, summary in LEGACY_RUNTIME_CLOSURE.items()
    },
)


CONSTRUCTION_STREAM_RESULTS = {}
AUDIT_STREAM_SUMMARIES = {}
for role, seeds, paths, expose_counts in (
    (
        "construction", FRESH_CONSTRUCTION_SEED,
        CONSTRUCTION_TEMPLATE_PATHS, True,
    ),
    (
        "audit", FRESH_AUDIT_SEED,
        SEALED_AUDIT_TEMPLATE_PATHS, False,
    ),
):
    print(
        f"\nEnsuring fresh {role} stream: "
        f"{FINAL_TEMPLATE_EVENTS:,} selected events per process"
    )
    for process in ("signal", "background"):
        role_results_so_far = (
            CONSTRUCTION_STREAM_RESULTS
            if role == "construction" else AUDIT_STREAM_SUMMARIES
        )
        if (
            not paths[process].exists()
            and role_results_so_far
            and any(
                summary["runtime_fingerprint"] != STREAM_RUNTIME_FINGERPRINT
                for summary in role_results_so_far.values()
            )
        ):
            raise RuntimeError(
                f"Refusing to start the missing {role} {process} cache "
                "under a runtime incompatible with its completed peer."
            )
        components = RECO_COMPONENTS[process]
        result = run_streamed_simulator_template(
            cache_path=paths[process],
            process=process,
            selected_checkpoints=TEMPLATE_SELECTED_CHECKPOINTS,
            batch_size=FRESH_SIMULATOR_BATCH_SIZE,
            seed=seeds[process],
            feature_names=FEATURES,
            feature_edges=FEATURE_DIAGNOSTIC_EDGES,
            log_q_edges=LOG_Q_EDGES,
            sample_batch=lambda n_events, rng, components=components: (
                sample_nominal_simulator_features(components, n_events, rng)
            ),
            preselection_ratio=evaluate_PRESEL_ratio,
            log_q_evaluator=evaluate_log_q,
            presel_ratio_cut=PRESEL_RATIO_CUT,
            recipe_fingerprint=f"{STREAM_RECIPE_FINGERPRINT}:{role}",
            runtime_fingerprint=STREAM_RUNTIME_FINGERPRINT,
            save_every_batches=TEMPLATE_SAVE_EVERY_BATCHES,
            progress_every_batches=TEMPLATE_PROGRESS_EVERY_BATCHES,
            expose_counts=expose_counts,
        )
        if role == "construction":
            CONSTRUCTION_STREAM_RESULTS[process] = result
        else:
            AUDIT_STREAM_SUMMARIES[process] = result
    role_results = (
        CONSTRUCTION_STREAM_RESULTS
        if role == "construction" else AUDIT_STREAM_SUMMARIES
    )
    role_runtime_fingerprints = {
        summary["runtime_fingerprint"] for summary in role_results.values()
    }
    if len(role_runtime_fingerprints) != 1:
        raise RuntimeError(
            f"The {role} signal/background caches were built with "
            "incompatible inference runtimes."
        )
    if (
        role == "construction"
        and STREAM_RUNTIME_FINGERPRINT not in role_runtime_fingerprints
        and any(not path.exists() for path in SEALED_AUDIT_TEMPLATE_PATHS.values())
    ):
        raise RuntimeError(
            "The complete construction cache was built under a different "
            "inference runtime, while no matching audit cache exists. "
            "Reacquire the recorded runtime or start new versioned streams."
        )


recorded_stream_runtimes = {
    summary["runtime_fingerprint"]
    for summary in (
        list(CONSTRUCTION_STREAM_RESULTS.values())
        + list(AUDIT_STREAM_SUMMARIES.values())
    )
}
if len(recorded_stream_runtimes) != 1:
    raise RuntimeError(
        "Construction and sealed-audit caches use incompatible inference "
        "runtimes and cannot support a joint coverage claim."
    )
STREAM_CACHE_RUNTIME_FINGERPRINT = next(iter(recorded_stream_runtimes))


SIM_CALIBRATION_SIGNAL_COUNTS_BY_CHECKPOINT = np.asarray(
    CONSTRUCTION_STREAM_RESULTS["signal"]["q_counts"], dtype=np.int64
)
SIM_CALIBRATION_BACKGROUND_COUNTS_BY_CHECKPOINT = np.asarray(
    CONSTRUCTION_STREAM_RESULTS["background"]["q_counts"], dtype=np.int64
)
SIM_CALIBRATION_SIGNAL_FEATURE_COUNTS_BY_CHECKPOINT = np.asarray(
    CONSTRUCTION_STREAM_RESULTS["signal"]["feature_counts"], dtype=np.int64
)
SIM_CALIBRATION_BACKGROUND_FEATURE_COUNTS_BY_CHECKPOINT = np.asarray(
    CONSTRUCTION_STREAM_RESULTS["background"]["feature_counts"], dtype=np.int64
)
checkpoint_denominator = TEMPLATE_SELECTED_CHECKPOINTS[:, None].astype(np.float64)
SIM_CALIBRATION_SIGNAL_PROBABILITY_BY_CHECKPOINT = (
    SIM_CALIBRATION_SIGNAL_COUNTS_BY_CHECKPOINT / checkpoint_denominator
)
SIM_CALIBRATION_BACKGROUND_PROBABILITY_BY_CHECKPOINT = (
    SIM_CALIBRATION_BACKGROUND_COUNTS_BY_CHECKPOINT / checkpoint_denominator
)
SIM_CALIBRATION_SIGNAL_COUNTS = SIM_CALIBRATION_SIGNAL_COUNTS_BY_CHECKPOINT[-1]
SIM_CALIBRATION_BACKGROUND_COUNTS = SIM_CALIBRATION_BACKGROUND_COUNTS_BY_CHECKPOINT[-1]
SIM_CALIBRATION_SIGNAL_PROBABILITY = SIM_CALIBRATION_SIGNAL_PROBABILITY_BY_CHECKPOINT[-1]
SIM_CALIBRATION_BACKGROUND_PROBABILITY = SIM_CALIBRATION_BACKGROUND_PROBABILITY_BY_CHECKPOINT[-1]
SIM_CALIBRATION_EVENTS = {
    process: int(CONSTRUCTION_STREAM_RESULTS[process]["selected_events"])
    for process in ("signal", "background")
}
construction_digest = hashlib.sha256()
for process in ("signal", "background"):
    construction_digest.update(
        CONSTRUCTION_STREAM_RESULTS[process]["fingerprint"].encode("ascii")
    )
CONSTRUCTION_SOURCE_FINGERPRINT = construction_digest.hexdigest()

print("Construction simulator events:", SIM_CALIBRATION_EVENTS)
print("Construction fingerprint:", CONSTRUCTION_SOURCE_FINGERPRINT)
print(
    "The new audit stream is complete and provenance-certified, but its q "
    "and feature laws remain unopened:",
    {process: result["fingerprint"] for process, result in AUDIT_STREAM_SUMMARIES.items()},
)

# Event networks are no longer needed after both streams are compressed. Free
# their Torch/ONNX allocations before training the large statistic flow.
reference_flow = None
ratio_models = None
PRESEL_model = None
PRESEL_scaler = None
event_sessions = None
model_proto = None
scaler = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released frozen Exercise 5 event networks.")


### Validate the compression over the full design interval

Exercise 5 checked one Asimov displacement. Here the unbinned and compressed expected statistics are compared for several generating and tested values across $[0,3]$. This validates the numerical acceleration before it is used half a million times.


In [ ]:
compression_validation = pd.DataFrame({
    "mu_true": compression_truth,
    "mu_test": compression_test,
    "t_unbinned": compression_unbinned,
    "t_compressed": compression_binned,
})
compression_validation["absolute_difference"] = (
    compression_validation["t_compressed"]
    - compression_validation["t_unbinned"]
)
compression_validation["relative_difference"] = np.divide(
    compression_validation["absolute_difference"],
    compression_validation["t_unbinned"],
    out=np.zeros(len(compression_validation)),
    where=compression_validation["t_unbinned"] > 1.0e-8,
)
display(compression_validation.style.format(precision=6).hide(axis="index"))
max_relative = float(compression_validation["relative_difference"].abs().max())
max_absolute = float(compression_validation["absolute_difference"].abs().max())
print(f"Maximum relative/absolute change: {max_relative:.3%} / {max_absolute:.4g}")
if max_relative > 5.0e-3 and max_absolute > 2.0e-2:
    raise RuntimeError(
        "The 512-bin compression is not accurate to 0.5% across the scan. "
        "Increase TOY_Q_BINS."
    )


## Vectorized pseudo-experiment fits

For compressed counts $n_j$, the hNDE-coordinate MLE $\widehat\nu$ solves

$$
0=\lambda_S-\sum_j n_j\frac{q_j}{1+\widehat\nu q_j},
\qquad \widehat\nu\geq0.
$$

JAX performs 16 bounded Newton steps for an entire toy batch, with a vectorized bisection fallback. For any tested hNDE coordinate $\nu_0$ it returns

$$
t(\nu_0)=2\left[(\nu_0-\widehat\nu)\lambda_S
-\sum_j n_j\left\{\log(1+\nu_0q_j)
-\log(1+\widehat\nu q_j)\right\}\right].
$$

The response-corrected statistic is obtained by passing $\nu_0=g(\mu)$. The toy generator separately records the physical parameter $\mu$, the parameter used to generate counts, and the tested likelihood coordinate. This separation is essential: hNDE toys use $g(\mu)$ for both generation and testing, whereas simulator toys generate at physical $\mu$ and test $g(\mu)$.


In [ ]:
COMPRESSED_Q_JAX = jnp.asarray(COMPRESSED_Q)


@jax.jit
def fit_compressed_toy_batch(counts, test_mu):
    counts = jnp.asarray(counts, dtype=jnp.float64)
    test_mu = jnp.asarray(test_mu, dtype=jnp.float64).reshape(-1)
    q_values = COMPRESSED_Q_JAX
    initial_mu = jnp.clip(
        (jnp.sum(counts, axis=1) - LAM_BKG) / LAM_SIG,
        0.0,
        TOY_MU_MAX,
    )
    score_at_zero = LAM_SIG - jnp.sum(counts * q_values, axis=1)

    def newton_step(_, mu):
        response = q_values / (1.0 + mu[:, None] * q_values)
        score = LAM_SIG - jnp.sum(counts * response, axis=1)
        information = jnp.sum(counts * response**2, axis=1)
        step = jnp.clip(
            score / jnp.maximum(information, 1.0e-12), -2.0, 2.0
        )
        return jnp.clip(mu - step, 0.0, TOY_MU_MAX)

    mu_hat = jax.lax.fori_loop(
        0, TOY_NEWTON_STEPS, newton_step, initial_mu
    )
    mu_hat = jnp.where(score_at_zero >= 0.0, 0.0, mu_hat)
    statistic = 2.0 * (
        (test_mu - mu_hat) * LAM_SIG
        - jnp.sum(
            counts * (
                jnp.log1p(test_mu[:, None] * q_values)
                - jnp.log1p(mu_hat[:, None] * q_values)
            ),
            axis=1,
        )
    )
    fitted_response = q_values / (1.0 + mu_hat[:, None] * q_values)
    fitted_score = LAM_SIG - jnp.sum(
        counts * fitted_response, axis=1
    )
    fitted_score = jnp.where(mu_hat == 0.0, 0.0, fitted_score)
    return mu_hat, jnp.maximum(statistic, 0.0), fitted_score


def fit_toy_batch_numpy(counts, test_mu):
    counts = np.asarray(counts)
    test_mu = np.asarray(test_mu, dtype=np.float64).reshape(-1)
    result = fit_compressed_toy_batch(counts, test_mu)
    mu_hat, t_mu, fitted_score = [
        np.asarray(values, dtype=np.float64) for values in result
    ]
    failed = (
        (mu_hat > 1.0e-10)
        & (mu_hat < TOY_MU_MAX - 1.0e-10)
        & (np.abs(fitted_score) > 1.0e-6)
    )
    if np.any(failed):
        # The score is monotone increasing in mu. A vectorized bisection
        # fallback makes rare Newton failures harmless without returning
        # to one-Minuit-fit-per-toy execution.
        failed_counts = counts[failed].astype(np.float64)
        lower = np.zeros(np.sum(failed), dtype=np.float64)
        upper = np.full(np.sum(failed), TOY_MU_MAX, dtype=np.float64)
        for _ in range(64):
            middle = 0.5 * (lower + upper)
            response = COMPRESSED_Q / (
                1.0 + middle[:, None] * COMPRESSED_Q
            )
            middle_score = LAM_SIG - np.sum(
                failed_counts * response, axis=1
            )
            move_lower = middle_score < 0.0
            lower = np.where(move_lower, middle, lower)
            upper = np.where(move_lower, upper, middle)
        repaired_mu = 0.5 * (lower + upper)
        mu_hat[failed] = repaired_mu
        failed_test_mu = test_mu[failed]
        repaired_t = 2.0 * (
            (failed_test_mu - repaired_mu) * LAM_SIG
            - np.sum(
                failed_counts
                * (
                    np.log1p(failed_test_mu[:, None] * COMPRESSED_Q)
                    - np.log1p(repaired_mu[:, None] * COMPRESSED_Q)
                ),
                axis=1,
            )
        )
        t_mu[failed] = np.maximum(repaired_t, 0.0)
        repaired_response = COMPRESSED_Q / (
            1.0 + repaired_mu[:, None] * COMPRESSED_Q
        )
        fitted_score[failed] = LAM_SIG - np.sum(
            failed_counts * repaired_response, axis=1
        )
    if np.any(mu_hat >= TOY_MU_MAX - 1.0e-10):
        raise RuntimeError(
            "A toy MLE reached TOY_MU_MAX; increase the fit bound."
        )
    return mu_hat, t_mu, fitted_score


# Compile once and check that the fitted score is small away from the boundary.
_rng = np.random.default_rng(SEED + 200)
_mu = _rng.uniform(*MU_RANGE, size=8)
_mean = (
    _mu[:, None] * LAM_SIG * HNDE_SIGNAL_PROBABILITY[None, :]
    + LAM_BKG * HNDE_BACKGROUND_PROBABILITY[None, :]
)
_counts = _rng.poisson(_mean)
_muhat, _tmu, _score = fit_toy_batch_numpy(_counts, _mu)
print("Toy-kernel smoke test:")
print("  mu_true:", np.round(_mu, 3))
print("  mu_hat: ", np.round(_muhat, 3))
print("  t_mu:  ", np.round(_tmu, 3))
print(f"  max interior score residual: {np.max(np.abs(_score)):.3e}")
toy_kernel_digest = hashlib.sha256()
for label, values in (
    ("counts", _counts), ("test_mu", _mu),
    ("mu_hat", _muhat), ("t_mu", _tmu), ("score", _score),
):
    update_array_digest(toy_kernel_digest, label, values)
jax_device = jax.devices()[0]
JAX_RUNTIME_ID = (
    f"jax={jax.__version__}|backend={jax.default_backend()}|"
    f"platform={jax_device.platform}|kind={jax_device.device_kind}|x64=True"
)
toy_kernel_digest.update(JAX_RUNTIME_ID.encode("utf-8"))
TOY_FIT_RUNTIME_FINGERPRINT = (
    "jax_toy_runtime_v1_" + toy_kernel_digest.hexdigest()[:24]
)
print(
    "Toy-fit recipe/runtime:", TOY_FIT_FINGERPRINT,
    JAX_RUNTIME_ID, TOY_FIT_RUNTIME_FINGERPRINT,
)
del _rng, _mu, _mean, _counts, _muhat, _tmu, _score


## 1. Learn the monotone pseudo-truth response $g(\mu)$

Regressing the finite-sample mean of $\widehat\nu$ would mix the population response with estimator bias and the $\widehat\nu\geq0$ boundary. Instead we compute the KL pseudo-truth directly from the construction simulator template. For compressed simulator means

$$
m_j^{\rm sim}(\mu)
=\mu\lambda_S P^{\rm sim}_{S,j}
+\lambda_B P^{\rm sim}_{B,j},
$$

$g(\mu)$ obeys the population score equation

$$
\lambda_S-\sum_j m_j^{\rm sim}(\mu)
\frac{q_j}{1+g(\mu)q_j}=0,
$$

unless the constrained optimum is $g(\mu)=0$. Positivity of the templates and of $q_j$ makes this response nondecreasing. We solve it on a dense physical-$\mu$ grid, validate the KKT condition, interpolate with a monotone PCHIP, and check the interpolation at every midpoint.

Crucially, the construction template alone defines $g$. The unused audit-event template remains sealed. We do not force $g(0)=0$ or $g(3)=3$: either constraint would erase precisely the simulator mismatch we are trying to represent.


In [ ]:
RESPONSE_MU_GRID = np.linspace(*MU_RANGE, RESPONSE_GRID_POINTS)
response_expected_counts = (
    RESPONSE_MU_GRID[:, None]
    * LAM_SIG
    * SIM_CALIBRATION_SIGNAL_PROBABILITY[None, :]
    + LAM_BKG * SIM_CALIBRATION_BACKGROUND_PROBABILITY[None, :]
)
response_grid, _, _ = fit_toy_batch_numpy(
    response_expected_counts, RESPONSE_MU_GRID
)
response_grid = np.asarray(response_grid, dtype=np.float64)
response_score = LAM_SIG - np.sum(
    response_expected_counts
    * COMPRESSED_Q[None, :]
    / (1.0 + response_grid[:, None] * COMPRESSED_Q[None, :]),
    axis=1,
)
response_interior = response_grid > 1.0e-10
response_kkt_violation = np.where(
    response_interior,
    np.abs(response_score),
    np.maximum(-response_score, 0.0),
)
if np.max(response_kkt_violation) > 1.0e-6:
    raise RuntimeError(
        "The pseudo-truth response does not satisfy the constrained "
        "population score equation."
    )
if np.min(np.diff(response_grid)) < -1.0e-10:
    raise RuntimeError("The fitted pseudo-truth response is not monotone.")
if response_grid.min() < 0.0 or response_grid.max() >= TOY_MU_MAX - 0.5:
    raise RuntimeError(
        "The pseudo-truth response lies outside the safe hNDE fit range."
    )

response_interpolator = PchipInterpolator(
    RESPONSE_MU_GRID, response_grid, extrapolate=False
)


def pseudo_truth_response(mu):
    """Map physical simulator truth to the frozen hNDE coordinate."""
    values = np.asarray(mu, dtype=np.float64)
    tolerance = 1.0e-12
    if (
        not np.isfinite(values).all()
        or np.any(values < MU_RANGE[0] - tolerance)
        or np.any(values > MU_RANGE[1] + tolerance)
    ):
        raise ValueError("Physical mu lies outside the response-map range.")
    result = np.asarray(
        response_interpolator(np.clip(values, *MU_RANGE)),
        dtype=np.float64,
    )
    return result


response_midpoints = 0.5 * (
    RESPONSE_MU_GRID[:-1] + RESPONSE_MU_GRID[1:]
)
midpoint_expected_counts = (
    response_midpoints[:, None]
    * LAM_SIG
    * SIM_CALIBRATION_SIGNAL_PROBABILITY[None, :]
    + LAM_BKG * SIM_CALIBRATION_BACKGROUND_PROBABILITY[None, :]
)
midpoint_exact, _, _ = fit_toy_batch_numpy(
    midpoint_expected_counts, response_midpoints
)
response_midpoint_error = float(np.max(np.abs(
    pseudo_truth_response(response_midpoints) - midpoint_exact
)))
if response_midpoint_error > 5.0e-5:
    raise RuntimeError(
        "The response-map interpolation is not accurate enough; increase "
        "RESPONSE_GRID_POINTS."
    )

response_digest = hashlib.sha256()
for values in (RESPONSE_MU_GRID, response_grid):
    response_digest.update(np.ascontiguousarray(values).view(np.uint8))
RESPONSE_MAP_FINGERPRINT = (
    "simulator_pseudotruth_v2_" + response_digest.hexdigest()[:16]
)


def response_corrected_test_statistic(counts, physical_mu):
    """Fit counts and evaluate t_mu^(g) at physical ``mu``."""
    return fit_toy_batch_numpy(
        counts, pseudo_truth_response(physical_mu)
    )


print(
    "Pseudo-truth response endpoints:",
    f"g(0)={response_grid[0]:.6f}, g(3)={response_grid[-1]:.6f}",
)
print(
    "Response displacement range:",
    f"[{np.min(response_grid - RESPONSE_MU_GRID):+.6f}, "
    f"{np.max(response_grid - RESPONSE_MU_GRID):+.6f}]",
)
print("Maximum response interpolation error:", response_midpoint_error)
print("Response-map fingerprint:", RESPONSE_MAP_FINGERPRINT)

fig, ax = plt.subplots(figsize=(7.0, 4.8))
ax.plot(RESPONSE_MU_GRID, response_grid, lw=2.4, label=r"$g(\mu)$")
ax.plot(MU_RANGE, MU_RANGE, "k--", lw=1.2, label="identity")
ax.set(
    xlim=MU_RANGE,
    xlabel=r"physical simulator truth $\mu$",
    ylabel=r"pseudo-true hNDE coordinate $g(\mu)$",
    title="Monotone simulator-to-hNDE response",
)
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
export_exercise11_figure(fig, "pseudo_truth_response_map")
plt.show()


## 2. Freeze the 64M construction law and diagnose convergence before training

Before the neural simulator correction is trained, we inspect the nested construction prefixes. Raw $L^1$ distance is reported, but it is not the decisive diagnostic: the previous run showed that a smaller $L^1$ perturbation can have a much larger coverage effect when it points along the likelihood score.

For every prefix $N$, we therefore recompute its population response $g_N(\mu)$ and evaluate the score of the final 64M construction law at that response,

$$
S_{64}(g_N;\mu)
=\lambda_S-\sum_j m_{64,j}(\mu)\frac{q_j}{1+g_N(\mu)q_j},
$$

with curvature

$$
I_{64}(g_N;\mu)
=\sum_j m_{64,j}(\mu)\frac{q_j^2}{[1+g_N(\mu)q_j]^2}.
$$

The dimensionless quantity $|S_{64}|/\sqrt{I_{64}}$ measures the dangerous response displacement in approximate standard-error units. Prefixes are nested and correlated; these curves are convergence diagnostics, not independent repetitions.

The current-runtime Exercise 5 parquet replay is also compared with the fresh construction in PRESEL acceptance, selected-feature histograms, $q$ moments, and response. Closeness is reported rather than enforced by a brittle statistical guard. Structural cache/count/fingerprint failures remain hard errors.


In [ ]:
def template_response_grid(signal_probability, background_probability):
    expected = (
        RESPONSE_MU_GRID[:, None]
        * LAM_SIG
        * np.asarray(signal_probability)[None, :]
        + LAM_BKG * np.asarray(background_probability)[None, :]
    )
    fitted, _, _ = fit_toy_batch_numpy(expected, RESPONSE_MU_GRID)
    return np.asarray(fitted, dtype=np.float64)


def standardized_score_drift(
    response_values, target_signal_probability, target_background_probability
):
    expected = (
        RESPONSE_MU_GRID[:, None]
        * LAM_SIG
        * np.asarray(target_signal_probability)[None, :]
        + LAM_BKG * np.asarray(target_background_probability)[None, :]
    )
    denominator = 1.0 + np.asarray(response_values)[:, None] * COMPRESSED_Q[None, :]
    score = LAM_SIG - np.sum(
        expected * COMPRESSED_Q[None, :] / denominator, axis=1
    )
    information = np.sum(
        expected * COMPRESSED_Q[None, :] ** 2 / denominator**2, axis=1
    )
    standardized = np.divide(
        np.abs(score), np.sqrt(information),
        out=np.full_like(score, np.inf), where=information > 0.0,
    )
    return standardized


PREFIX_RESPONSE_GRIDS = []
PREFIX_RESPONSE_FINGERPRINTS = []
convergence_rows = []
log_q_bin = np.log(COMPRESSED_Q)
for checkpoint_index, selected_events in enumerate(TEMPLATE_SELECTED_CHECKPOINTS):
    signal_probability = SIM_CALIBRATION_SIGNAL_PROBABILITY_BY_CHECKPOINT[
        checkpoint_index
    ]
    background_probability = SIM_CALIBRATION_BACKGROUND_PROBABILITY_BY_CHECKPOINT[
        checkpoint_index
    ]
    prefix_response = template_response_grid(
        signal_probability, background_probability
    )
    PREFIX_RESPONSE_GRIDS.append(prefix_response)
    digest = hashlib.sha256()
    update_array_digest(digest, "mu", RESPONSE_MU_GRID)
    update_array_digest(digest, "g", prefix_response)
    PREFIX_RESPONSE_FINGERPRINTS.append(
        f"construction_prefix_{int(selected_events)}_{digest.hexdigest()[:16]}"
    )
    score_drift = standardized_score_drift(
        prefix_response,
        SIM_CALIBRATION_SIGNAL_PROBABILITY,
        SIM_CALIBRATION_BACKGROUND_PROBABILITY,
    )
    convergence_rows.append({
        "source": "fresh nested construction",
        "selected_per_process": int(selected_events),
        "signal_L1_to_64M": float(np.sum(np.abs(
            signal_probability - SIM_CALIBRATION_SIGNAL_PROBABILITY
        ))),
        "background_L1_to_64M": float(np.sum(np.abs(
            background_probability - SIM_CALIBRATION_BACKGROUND_PROBABILITY
        ))),
        "max_abs_g_minus_g64": float(np.max(np.abs(
            prefix_response - response_grid
        ))),
        "max_standardized_score_drift": float(np.max(score_drift)),
        "g_0": float(prefix_response[0]),
        "g_3": float(prefix_response[-1]),
        "signal_mean_log_q": float(np.sum(signal_probability * log_q_bin)),
        "background_mean_log_q": float(np.sum(background_probability * log_q_bin)),
    })
PREFIX_RESPONSE_GRIDS = np.asarray(PREFIX_RESPONSE_GRIDS)


legacy_response_grid = template_response_grid(
    LEGACY_EXERCISE5_SIGNAL_PROBABILITY,
    LEGACY_EXERCISE5_BACKGROUND_PROBABILITY,
)
legacy_score_drift = standardized_score_drift(
    legacy_response_grid,
    SIM_CALIBRATION_SIGNAL_PROBABILITY,
    SIM_CALIBRATION_BACKGROUND_PROBABILITY,
)
convergence_rows.insert(0, {
    "source": "saved Exercise 5 legacy",
    "selected_per_process": int(LEGACY_EXERCISE5_SIGNAL_COUNTS.sum()),
    "signal_L1_to_64M": float(np.sum(np.abs(
        LEGACY_EXERCISE5_SIGNAL_PROBABILITY - SIM_CALIBRATION_SIGNAL_PROBABILITY
    ))),
    "background_L1_to_64M": float(np.sum(np.abs(
        LEGACY_EXERCISE5_BACKGROUND_PROBABILITY - SIM_CALIBRATION_BACKGROUND_PROBABILITY
    ))),
    "max_abs_g_minus_g64": float(np.max(np.abs(legacy_response_grid - response_grid))),
    "max_standardized_score_drift": float(np.max(legacy_score_drift)),
    "g_0": float(legacy_response_grid[0]),
    "g_3": float(legacy_response_grid[-1]),
    "signal_mean_log_q": float(np.sum(LEGACY_EXERCISE5_SIGNAL_PROBABILITY * log_q_bin)),
    "background_mean_log_q": float(np.sum(LEGACY_EXERCISE5_BACKGROUND_PROBABILITY * log_q_bin)),
})
template_convergence_table = pd.DataFrame(convergence_rows)
display(template_convergence_table.style.format({
    "selected_per_process": "{:,.0f}",
    "signal_L1_to_64M": "{:.6f}",
    "background_L1_to_64M": "{:.6f}",
    "max_abs_g_minus_g64": "{:.6f}",
    "max_standardized_score_drift": "{:.4f}",
    "g_0": "{:.6f}",
    "g_3": "{:.6f}",
    "signal_mean_log_q": "{:.6f}",
    "background_mean_log_q": "{:.6f}",
}).hide(axis="index"))


legacy_acceptance_rows = []
feature_closure_rows = []
q_runtime_closure_rows = []
for process, construction_feature_counts in (
    ("signal", SIM_CALIBRATION_SIGNAL_FEATURE_COUNTS_BY_CHECKPOINT[-1]),
    ("background", SIM_CALIBRATION_BACKGROUND_FEATURE_COUNTS_BY_CHECKPOINT[-1]),
):
    legacy = LEGACY_RUNTIME_CLOSURE[process]
    legacy_acceptance_rows.append({
        "process": process,
        "source": "Exercise 5 eval parquet (current runtime)",
        "generated_or_partition_events": legacy["partition_events"],
        "selected_events": legacy["selected_events"],
        "PRESEL_acceptance": legacy["selected_events"] / legacy["partition_events"],
    })
    stream = CONSTRUCTION_STREAM_RESULTS[process]
    legacy_acceptance_rows.append({
        "process": process,
        "source": "fresh analytic construction stream",
        "generated_or_partition_events": stream["generated_events"],
        "selected_events": stream["selected_events"],
        "PRESEL_acceptance": stream["acceptance"],
    })
    legacy_feature_probability = (
        legacy["feature_counts"] / legacy["selected_events"]
    )
    construction_feature_probability = (
        construction_feature_counts / FINAL_TEMPLATE_EVENTS
    )
    construction_q_probability = (
        SIM_CALIBRATION_SIGNAL_PROBABILITY
        if process == "signal"
        else SIM_CALIBRATION_BACKGROUND_PROBABILITY
    )
    legacy_q_probability = legacy["q_counts"] / legacy["selected_events"]
    q_runtime_closure_rows.append({
        "process": process,
        "Exercise5_runtime_vs_fresh64M_q_L1": float(np.sum(np.abs(
            legacy_q_probability - construction_q_probability
        ))),
        "Exercise5_runtime_mean_log_q": float(np.sum(
            legacy_q_probability * log_q_bin
        )),
        "fresh64M_mean_log_q": float(np.sum(
            construction_q_probability * log_q_bin
        )),
    })
    for feature_index, feature in enumerate(FEATURES):
        feature_closure_rows.append({
            "process": process,
            "feature": feature,
            "histogram_L1": float(np.sum(np.abs(
                legacy_feature_probability[feature_index]
                - construction_feature_probability[feature_index]
            ))),
            "legacy_tail_fraction": float(
                legacy_feature_probability[feature_index, 0]
                + legacy_feature_probability[feature_index, -1]
            ),
            "construction_tail_fraction": float(
                construction_feature_probability[feature_index, 0]
                + construction_feature_probability[feature_index, -1]
            ),
        })
preselection_closure_table = pd.DataFrame(legacy_acceptance_rows)
feature_closure_table = pd.DataFrame(feature_closure_rows)
q_runtime_closure_table = pd.DataFrame(q_runtime_closure_rows)
display(preselection_closure_table.style.format({
    "generated_or_partition_events": "{:,.0f}",
    "selected_events": "{:,.0f}",
    "PRESEL_acceptance": "{:.6%}",
}).hide(axis="index"))
display(feature_closure_table.style.format(precision=6).hide(axis="index"))
display(q_runtime_closure_table.style.format(precision=6).hide(axis="index"))


fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))
for index, selected_events in enumerate(TEMPLATE_SELECTED_CHECKPOINTS):
    axes[0].plot(
        RESPONSE_MU_GRID,
        PREFIX_RESPONSE_GRIDS[index] - response_grid,
        lw=1.7,
        label=f"{int(selected_events):,}",
    )
axes[0].plot(
    RESPONSE_MU_GRID, legacy_response_grid - response_grid,
    color="black", ls="--", lw=1.5, label="Exercise 5 legacy",
)
axes[0].axhline(0.0, color="0.5", lw=1.0)
axes[0].set(
    xlabel=r"physical truth $\mu$",
    ylabel=r"$g_N(\mu)-g_{64\mathrm{M}}(\mu)$",
    title="Nested response-map convergence",
)
axes[0].grid(alpha=.2)
axes[0].legend(fontsize=7, ncol=2)

for process, marker in (("signal", "o"), ("background", "s")):
    subset = feature_closure_table[feature_closure_table["process"] == process]
    axes[1].plot(
        subset["feature"], subset["histogram_L1"], marker=marker,
        lw=1.8, label=process,
    )
axes[1].set(
    ylabel="Exercise 5 vs fresh 64M feature-histogram $L^1$",
    title="Generator-version feature closure",
)
axes[1].grid(alpha=.2)
axes[1].legend()
fig.tight_layout()
export_exercise11_figure(fig, "streamed_template_response_and_feature_closure")
plt.show()

assert np.isclose(SIM_CALIBRATION_SIGNAL_PROBABILITY.sum(), 1.0)
assert np.isclose(SIM_CALIBRATION_BACKGROUND_PROBABILITY.sum(), 1.0)
print(
    "Sealed audit provenance only:",
    {process: summary["fingerprint"] for process, summary in AUDIT_STREAM_SUMMARIES.items()},
)


## 3. Generate 500,000 response-matched hNDE toys

Draw physical design points $\mu\sim U(0,3)$, map them to $g(\mu)$, generate the inexpensive hNDE toy at that surrogate coordinate, and evaluate the numerator at the same coordinate. Thus these toys learn the central response-matched reference law

$$
p_{\rm H}^{(g)}(t\mid\mu),
\qquad
\mathcal D_{\rm H}\sim p_{\rm H}(\cdot\mid g(\mu)),
\qquad
t=t_\mu^{(g)}(\mathcal D_{\rm H}).
$$

The context stored for every flow and classifier remains the physical $\mu$, not $g(\mu)$. The cache records all three coordinates and aborts if a stale identity-response shard is encountered.


In [ ]:
hnde_toys = run_cached_toy_ensemble(
    cache_dir=CACHE_DIR / f"hnde_uniform_{N_HNDE_TOYS}",
    n_toys=N_HNDE_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 300,
    mu_range=MU_RANGE,
    signal_probability=HNDE_SIGNAL_PROBABILITY,
    background_probability=HNDE_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
    fit_runtime_fingerprint=TOY_FIT_RUNTIME_FINGERPRINT,
    generation_mu_transform=pseudo_truth_response,
    test_mu_transform=pseudo_truth_response,
    generation_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
    test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
)
expected_hnde_coordinate = pseudo_truth_response(hnde_toys["mu"])
if not np.allclose(
    hnde_toys["generation_mu"], expected_hnde_coordinate,
    rtol=0.0, atol=2.0e-6,
) or not np.allclose(
    hnde_toys["test_mu"], expected_hnde_coordinate,
    rtol=0.0, atol=2.0e-6,
):
    raise RuntimeError("The response-matched hNDE cache has stale coordinates.")
print(pd.DataFrame({
    "mu": hnde_toys["mu"],
    "g_mu": hnde_toys["test_mu"],
    "mu_hat": hnde_toys["mu_hat"],
    "t_mu": hnde_toys["t_mu"],
    "n_events": hnde_toys["n_events"],
}).describe(percentiles=[.01, .5, .95, .99]).to_string())
print(
    "Maximum fitted-score residual:",
    f"{np.max(np.abs(hnde_toys['fitted_score'])):.3e}",
)

split_rng = np.random.default_rng(SEED + 301)
toy_order = split_rng.permutation(N_HNDE_TOYS)
flow_indices = toy_order[:N_FLOW_TOYS]
ratio1_indices = toy_order[N_FLOW_TOYS:]
flow_mu = hnde_toys["mu"][flow_indices].astype(np.float32)
flow_y = np.log(
    hnde_toys["t_mu"][flow_indices].astype(np.float64) + T_OFFSET
).astype(np.float32)
ratio1_mu = hnde_toys["mu"][ratio1_indices].astype(np.float32)
ratio1_y_positive = np.log(
    hnde_toys["t_mu"][ratio1_indices].astype(np.float64) + T_OFFSET
).astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
axes[0].hexbin(
    hnde_toys["mu"], hnde_toys["mu_hat"],
    gridsize=80, bins="log", mincnt=1, cmap="viridis",
)
response_plot_grid = np.linspace(*MU_RANGE, 400)
axes[0].plot(
    response_plot_grid, pseudo_truth_response(response_plot_grid),
    "w--", lw=1.5, label=r"$g(\mu)$",
)
axes[0].set(
    xlabel=r"physical $\mu$", ylabel=r"$\widehat\nu$",
    title="Response-matched hNDE pseudo-experiments",
)
axes[0].legend()
for low, high, color in [(0.0,.25,"C0"),(.75,1.0,"C1"),(1.75,2.0,"C2"),(2.75,3.0,"C3")]:
    mask = (hnde_toys["mu"] >= low) & (hnde_toys["mu"] < high)
    values = hnde_toys["t_mu"][mask]
    edges = np.linspace(0, np.quantile(values, .995), 60)
    axes[1].hist(values, bins=edges, density=True, histtype="step", lw=1.8,
                 color=color, label=rf"$\mu\in[{low:g},{high:g})$")
x = np.linspace(0.001, axes[1].get_xlim()[1], 400)
axes[1].plot(x, chi2.pdf(x, df=1), "k--", lw=1.3, label=r"$\chi^2_1$")
axes[1].set(
    xlabel=r"$t_{\mu}^{(g)}$", ylabel="Density",
    title="Response matching does not assume a pivot",
)
axes[1].legend(fontsize=8)
axes[1].set_yscale('log')
fig.tight_layout()
export_exercise11_figure(fig, "hnde_amortized_toys")
plt.show()


## 4. Train the conditional quadratic-spline reference

The first 400,000 response-matched hNDE toys train a conditional rational-quadratic-spline density in $y=\log(t_\mu^{(g)}+\epsilon)$. The remaining 100,000 toys are disjoint and are reserved for the first density-ratio correction.

The scalar flow is conditioned on physical $\mu$. Its target includes the non-Gaussian structure left after response matching; no Wilks or Wald approximation is imposed.


In [ ]:
statistic_flow = train_spline_flow(
    flow_y[:, None],
    context=flow_mu[:, None],
    checkpoint=MODEL_DIR / "q_phi_y_given_mu.pt",
    model_config=STATISTIC_FLOW_MODEL_CONFIG,
    training_config=STATISTIC_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 400,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
flow_config_mismatch = {
    name: (statistic_flow["config"].get(name), expected)
    for name, expected in STATISTIC_FLOW_MODEL_CONFIG.items()
    if statistic_flow["config"].get(name) != expected
}
if flow_config_mismatch:
    raise RuntimeError(
        "The loaded statistic-flow checkpoint has stale architecture: "
        f"{flow_config_mismatch}. Bump BASE_RUN_TAG."
    )
print("Conditional statistic flow:", statistic_flow["checkpoint"])


## 5. First matched correction: spline $\rightarrow$ response-matched hNDE toys

At each physical $\mu_i$, a positive example is the held-out hNDE statistic $y_i$ and its matched negative example is drawn from the conditional spline at that same $\mu_i$. Ordinary BCE therefore estimates the residual density ratio without proposal-prior weights. Paired group splitting keeps the positive and negative members of each matched pair on the same side of the train/validation boundary.


In [ ]:
Y_MIN = float(np.log(T_OFFSET))
ratio1_y_negative, ratio1_rejection = sample_truncated_spline_flow(
    statistic_flow,
    ratio1_mu[:, None],
    lower_bound=Y_MIN,
    seed=SEED + 500,
)
ratio1_positive = np.column_stack([ratio1_mu, ratio1_y_positive])
ratio1_negative = np.column_stack([ratio1_mu, ratio1_y_negative])
paired_ids_1 = np.arange(N_RATIO1_TOYS, dtype=np.int64)
ratio1_ensemble = []
for member in range(RATIO_ENSEMBLE_SIZE):
    print("\n" + "=" * 76)
    print(f"Training hNDE residual member {member + 1}/{RATIO_ENSEMBLE_SIZE}")
    print("=" * 76)
    ratio1_ensemble.append(
        train_ratio_classifier(
            ratio1_positive,
            ratio1_negative,
            checkpoint=RATIO1_MODEL_DIR / f"r1_member{member}.pt",
            model_config=CORRECTION_MODEL_CONFIG,
            training_config=CORRECTION_TRAINING_CONFIG,
            device=device,
            seed=SEED + 510 + 100 * member,
            load_if_available=LOAD_IF_AVAILABLE,
            paired_group_ids=paired_ids_1,
            verify_checkpoint_data=True,
        )
    )
print(f"Unphysical flow-reference rejection fraction: {ratio1_rejection:.4%}")
assert all(
    pack["history"].get("split_strategy") == "paired_groups"
    for pack in ratio1_ensemble
)

fig, ax = plt.subplots(figsize=(7.0, 4.5))
for member, pack in enumerate(ratio1_ensemble):
    history = pack.get("history", {})
    ax.plot(history.get("validation", []), label=f"member {member}")
ax.set(xlabel="Epoch", ylabel="Validation BCE",
       title="First conditional-ratio ensemble")
ax.grid(alpha=.25)
ax.legend(ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "first_ratio_training")
plt.show()


## 6. Normalize the response-matched hNDE density and compute $F_{\rm H}^{(g)}$

The first run exposed an important numerical trap. Increasing $Y_{\max}$ while holding the number of grid points fixed coarsened the grid, eventually producing an impossible raw-flow mass above one. The old loop also tested the density while its endpoint was still inside the learned spline region, where a rational-quadratic spline need not have a monotone tail.

The replacement below is self-adjusting:

1. it places $Y_{\max}$ beyond the standardized spline tail boundary, where every scalar `nflows` transform is exactly the identity and the raw-flow tail is an analytic standard-normal tail;
2. when support expands, it increases the point count to preserve the original $\Delta y$;
3. it doubles a nested grid until full/half-grid quantiles and integrals agree;
4. it compares numerical raw-flow mass with the exact normal-tail mass;
5. it uses the bounded classifier log-odds to place a conservative analytic upper bound on omitted hybrid tail mass.

All guards now change the relevant parameter before retrying and have finite hard caps. Failure raises an actionable exception; it is no longer possible to print a failed diagnostic and silently continue.


In [ ]:
MU_DENSITY_GRID = np.linspace(*MU_RANGE, QUADRATURE_MU_POINTS)
data_driven_y_max = float(max(
    5.0,
    np.quantile(np.concatenate([flow_y, ratio1_y_positive]), .99999) + 1.5,
))
target_y_mean = float(statistic_flow["target_scaler"].mean[0])
target_y_std = float(statistic_flow["target_scaler"].std[0])
spline_tail_bound = float(
    statistic_flow["config"]["spline_tail_bound"]
)

if target_y_std <= 0.0:
    raise RuntimeError("The statistic-flow target scale is not positive.")

Y_SUPPORT_BUFFER = 1.0
Y_UPPER_TAIL_MASS_TARGET = 1.0e-5
Y_QUANTILE_STABILITY_TARGET = 1.0e-2
Y_HALF_INTEGRAL_TARGET = 1.0e-3
Y_RAW_MASS_TARGET = 1.0e-3
Y_SUPPORT_MAX_EXPANSIONS = 3
Y_RESOLUTION_MAX_REFINEMENTS = 3
Y_GRID_MAX_POINTS = 32_769

# Preserve the original nominal spacing even if analytic tail certification
# requires a larger support. Use an even number of intervals so Y_GRID[::2]
# is a nested half-resolution grid with the same endpoints.
base_dy = (
    data_driven_y_max - Y_MIN
) / (QUADRATURE_Y_POINTS - 1)
Y_MAX = float(max(
    data_driven_y_max,
    target_y_mean + (spline_tail_bound + 2.0) * target_y_std,
))

quadrature_converged = False
for support_attempt in range(Y_SUPPORT_MAX_EXPANSIONS + 1):
    intervals = int(np.ceil((Y_MAX - Y_MIN) / base_dy))
    intervals += intervals % 2
    tail_requires_expansion = False

    for refinement in range(Y_RESOLUTION_MAX_REFINEMENTS + 1):
        n_points = intervals + 1
        if n_points > Y_GRID_MAX_POINTS:
            raise RuntimeError(
                f"Adaptive Y_GRID requires {n_points:,} points, above the "
                f"hard cap {Y_GRID_MAX_POINTS:,}. Increase the cap only "
                "after inspecting the statistic-flow boundary spike."
            )
        Y_GRID = np.linspace(Y_MIN, Y_MAX, n_points)
        flow_physical_grid = conditional_density_grid(
            statistic_flow,
            [],
            MU_DENSITY_GRID,
            Y_GRID,
        )
        hybrid1_grid = conditional_density_grid(
            statistic_flow,
            [ratio1_ensemble],
            MU_DENSITY_GRID,
            Y_GRID,
            max_abs_log_ratio=LOG_RATIO_CLIP,
        )

        z_min = (Y_MIN - target_y_mean) / target_y_std
        z_max = (Y_MAX - target_y_mean) / target_y_std
        if z_min > -spline_tail_bound or z_max < spline_tail_bound:
            raise RuntimeError(
                "Y_GRID endpoints do not both lie in the exact linear "
                "tails of the statistic spline."
            )
        exact_captured_flow_mass = float(
            norm.cdf(z_max) - norm.cdf(z_min)
        )
        exact_physical_flow_mass = float(norm.sf(z_min))
        exact_raw_upper_tail = float(norm.sf(z_max))
        numerical_flow_mass = np.exp(
            flow_physical_grid["log_normalization"]
        )
        raw_mass_error = float(np.max(np.abs(
            numerical_flow_mass - exact_captured_flow_mass
        )))

        coarse_y_grid = Y_GRID[::2]
        coarse_flow_mass = trapezoid(
            flow_physical_grid["density"][:, ::2],
            x=coarse_y_grid,
            axis=1,
        )
        coarse_hybrid_mass = trapezoid(
            hybrid1_grid["density"][:, ::2],
            x=coarse_y_grid,
            axis=1,
        )
        flow_half_integral_error = float(np.max(np.abs(
            coarse_flow_mass - 1.0
        )))
        hybrid_half_integral_error = float(np.max(np.abs(
            coarse_hybrid_mass - 1.0
        )))
        coarse_hybrid_cdf = cumulative_trapezoid(
            hybrid1_grid["density"][:, ::2],
            x=coarse_y_grid,
            axis=1,
            initial=0.0,
        ) / coarse_hybrid_mass[:, None]
        hybrid1_quantile_y = conditional_quantiles(
            hybrid1_grid["cdf"], Y_GRID, QUANTILE_LEVELS
        )
        coarse_hybrid1_quantile_y = conditional_quantiles(
            coarse_hybrid_cdf, coarse_y_grid, QUANTILE_LEVELS
        )
        y_resolution_shift = float(np.max(np.abs(
            coarse_hybrid1_quantile_y - hybrid1_quantile_y
        )))

        partial_hybrid_normalization = np.exp(
            hybrid1_grid["log_normalization"]
        )
        hybrid_upper_tail_bound = float(
            np.exp(LOG_RATIO_CLIP)
            * exact_raw_upper_tail
            / np.min(partial_hybrid_normalization)
        )
        print(
            f"Y attempt {support_attempt + 1}.{refinement + 1}: "
            f"Y_MAX={Y_MAX:.4f}, z_max={z_max:.3f}, "
            f"points={n_points:,}, max_dy={np.max(np.diff(Y_GRID)):.3e}; "
            f"q_shift={y_resolution_shift:.3e}, "
            f"half_mass(raw/hybrid)={flow_half_integral_error:.3e}/"
            f"{hybrid_half_integral_error:.3e}, "
            f"raw_exact_error={raw_mass_error:.3e}, "
            f"hybrid_tail_bound={hybrid_upper_tail_bound:.3e}"
        )

        if hybrid_upper_tail_bound > Y_UPPER_TAIL_MASS_TARGET:
            tail_requires_expansion = True
            break
        resolution_is_stable = (
            y_resolution_shift <= Y_QUANTILE_STABILITY_TARGET
            and flow_half_integral_error <= Y_HALF_INTEGRAL_TARGET
            and hybrid_half_integral_error <= Y_HALF_INTEGRAL_TARGET
            and raw_mass_error <= Y_RAW_MASS_TARGET
        )
        if resolution_is_stable:
            quadrature_converged = True
            break
        intervals *= 2

    if quadrature_converged:
        break
    if tail_requires_expansion:
        if support_attempt == Y_SUPPORT_MAX_EXPANSIONS:
            raise RuntimeError(
                "The analytic hybrid upper-tail bound did not converge. "
                "Inspect the first-ratio tail or increase the finite support."
            )
        Y_MAX += target_y_std
        continue
    raise RuntimeError(
        "Y quadrature did not pass the nested-grid stability checks after "
        f"{Y_RESOLUTION_MAX_REFINEMENTS} refinements."
    )

if not quadrature_converged:
    raise RuntimeError("Adaptive Y quadrature terminated without convergence.")
if (len(Y_GRID) - 1) % 2 or Y_GRID[::2][-1] != Y_MAX:
    raise RuntimeError("The final full/half Y grids are not exactly nested.")

hybrid1_quantile_t = np.maximum(
    np.exp(hybrid1_quantile_y) - T_OFFSET, 0.0
)
hybrid1_log_normalization_truncated = (
    hybrid1_grid["log_normalization"]
    - np.log(exact_physical_flow_mass)
)

# Rejection sampling removes only y < Y_MIN. Because Y_MIN lies in the exact
# linear tail, C_phi is analytic and cannot exceed one through quadrature bias.
flow_physical_mass = np.full(
    len(MU_DENSITY_GRID), exact_physical_flow_mass, dtype=np.float64
)
ratio1_acceptance = np.full(
    len(ratio1_mu), exact_physical_flow_mass, dtype=np.float64
)
quadrature_rejection = 1.0 - (
    len(ratio1_acceptance) / np.sum(1.0 / ratio1_acceptance)
)

hybrid1_edge_ratio = float(np.max(
    hybrid1_grid["density"][:, -1]
    / np.max(hybrid1_grid["density"], axis=1)
))
upper_buffer_index = int(np.searchsorted(
    Y_GRID, Y_MAX - Y_SUPPORT_BUFFER
))
hybrid1_upper_buffer_mass = float(np.max(
    1.0 - hybrid1_grid["cdf"][:, upper_buffer_index]
))
print(
    "Final Y-grid support/points/max spacing:",
    f"[{Y_MIN:.4f}, {Y_MAX:.4f}] / {len(Y_GRID):,} / "
    f"{np.max(np.diff(Y_GRID)):.3e}",
)
print(
    "Analytic physical flow mass / omitted raw/hybrid upper tail:",
    exact_physical_flow_mass,
    exact_raw_upper_tail,
    hybrid_upper_tail_bound,
)
print(
    "Final hybrid edge ratio / last-unit mass (diagnostic only):",
    hybrid1_edge_ratio,
    hybrid1_upper_buffer_mass,
)
print(
    "Flow rejection: observed / analytic context-matched =",
    f"{ratio1_rejection:.4%} / {quadrature_rejection:.4%}",
)
if abs(ratio1_rejection - quadrature_rejection) > 5.0e-3:
    raise RuntimeError(
        "Rejection sampling and the analytic physical-support mass disagree."
    )
print(
    "log Z1_tilde(mu) quantiles:",
    np.quantile(
        hybrid1_log_normalization_truncated, [0, .01, .5, .99, 1]
    ),
)
print(
    "r1 quadrature logit range / clipped fraction:",
    hybrid1_grid["ratio_log_range"][0],
    f"{hybrid1_grid['ratio_clip_fraction'][0]:.4%}",
)
print(
    "Full/half y-grid maximum quantile shift / fixed tolerance:",
    y_resolution_shift,
    Y_QUANTILE_STABILITY_TARGET,
)

if (
    np.max(np.abs(hybrid1_grid["cdf"][:, 0])) > 1.0e-10
    or np.max(np.abs(hybrid1_grid["cdf"][:, -1] - 1.0)) > 1.0e-10
    or np.min(np.diff(hybrid1_grid["cdf"], axis=1)) < -1.0e-10
):
    raise RuntimeError("F_H^(g) is not a valid conditional CDF.")


## 7. Generate simulator-calibration pseudo-experiments

Simulator toys generate counts at physical $\mu$ under the construction template but evaluate the response-corrected numerator at $g(\mu)$. The main proposal is uniform on $[0,3]$. Because the previous auditor resolved the largest residual error at low $\mu$, we add a second matched proposal on $[0,0.75]$.

This oversampling does not change the conditional ratio target. Positive simulator PIT values and negative uniform PIT values carry exactly the same sampled $\mu_i$, so the nonuniform proposal density cancels from the classifier odds. It only allocates more calibration statistics where the earlier conditional estimate was weakest.


In [ ]:
simulator_calibration_uniform_toys = run_cached_toy_ensemble(
    cache_dir=(
        PIT_CACHE_DIR
        / f"simulator_construction_uniform_{N_SIMULATOR_CALIBRATION_TOYS}"
    ),
    n_toys=N_SIMULATOR_CALIBRATION_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 700,
    mu_range=MU_RANGE,
    signal_probability=SIM_CALIBRATION_SIGNAL_PROBABILITY,
    background_probability=SIM_CALIBRATION_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
    fit_runtime_fingerprint=TOY_FIT_RUNTIME_FINGERPRINT,
    test_mu_transform=pseudo_truth_response,
    test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
)

simulator_calibration_low_mu_toys = run_cached_toy_ensemble(
    cache_dir=(
        PIT_CACHE_DIR
        / f"simulator_construction_low_mu_{N_SIMULATOR_LOW_MU_TOYS}"
    ),
    n_toys=N_SIMULATOR_LOW_MU_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 701,
    mu_range=LOW_MU_CALIBRATION_RANGE,
    signal_probability=SIM_CALIBRATION_SIGNAL_PROBABILITY,
    background_probability=SIM_CALIBRATION_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
    fit_runtime_fingerprint=TOY_FIT_RUNTIME_FINGERPRINT,
    test_mu_transform=pseudo_truth_response,
    test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
)

simulator_calibration_toys = {
    name: np.concatenate([
        simulator_calibration_uniform_toys[name],
        simulator_calibration_low_mu_toys[name],
    ])
    for name in simulator_calibration_uniform_toys
}
N_SIMULATOR_CALIBRATION_TOTAL = len(simulator_calibration_toys["mu"])
expected_test_mu = pseudo_truth_response(
    simulator_calibration_toys["mu"]
)
if not np.allclose(
    simulator_calibration_toys["generation_mu"],
    simulator_calibration_toys["mu"],
    rtol=0.0,
    atol=2.0e-6,
) or not np.allclose(
    simulator_calibration_toys["test_mu"],
    expected_test_mu,
    rtol=0.0,
    atol=2.0e-6,
):
    raise RuntimeError(
        "The simulator calibration cache does not use physical generation "
        "and response-corrected testing."
    )
print(pd.DataFrame({
    "mu": simulator_calibration_toys["mu"],
    "g_mu": simulator_calibration_toys["test_mu"],
    "mu_hat": simulator_calibration_toys["mu_hat"],
    "mu_hat_minus_g": (
        simulator_calibration_toys["mu_hat"]
        - simulator_calibration_toys["test_mu"]
    ),
    "t_mu_g": simulator_calibration_toys["t_mu"],
}).describe(percentiles=[.01, .5, .95, .99]).to_string())

calibration_mu = simulator_calibration_toys["mu"].astype(np.float32)
calibration_y = np.log(
    simulator_calibration_toys["t_mu"].astype(np.float64) + T_OFFSET
)
below_y_grid = float(np.mean(calibration_y < Y_MIN))
above_y_grid = float(np.mean(calibration_y > Y_MAX))
print(
    "Simulator calibration toys outside Y_GRID (lower/upper):",
    f"{below_y_grid:.6%} / {above_y_grid:.6%}",
)
if below_y_grid > 0.0 or above_y_grid > 0.0:
    raise RuntimeError(
        "Simulator calibration statistics lie outside the certified Y_GRID. "
        "Increase its support and rerun before training the PIT ratio; "
        "clipping would create an artificial atom at u=0 or u=1."
    )


## 8. Learn the residual simulator ratio in the response-matched PIT coordinate

Map every simulator-calibration statistic through $F_{\rm H}^{(g)}$. Response matching should make this discrepancy appreciably smaller than in the identity-response run, but no equality is assumed. A matched classifier compares the simulator $u_0$ with a uniform draw at the same physical $\mu$ and estimates the entire residual conditional density, not only its 95% quantile.


In [ ]:
calibration_u0_raw = conditional_cdf_values(
    hybrid1_grid["cdf"],
    MU_DENSITY_GRID,
    Y_GRID,
    calibration_mu,
    calibration_y,
)
exact_boundary_fraction = float(np.mean(
    (calibration_u0_raw <= 0.0) | (calibration_u0_raw >= 1.0)
))
print(f"Exact PIT-boundary fraction: {exact_boundary_fraction:.6%}")
if exact_boundary_fraction > 0.0:
    raise RuntimeError(
        "Interior simulator PIT values reached exactly 0 or 1. "
        "Inspect CDF support/interpolation before training."
    )
calibration_u0 = np.clip(
    calibration_u0_raw, PIT_EPS, 1.0 - PIT_EPS
).astype(np.float32)
pit_clipped_fraction = float(np.mean(
    (calibration_u0_raw < PIT_EPS)
    | (calibration_u0_raw > 1.0 - PIT_EPS)
))
print(
    f"PIT values moved by the numerical {PIT_EPS:g} clip: "
    f"{pit_clipped_fraction:.6%}"
)

pit_rng = np.random.default_rng(SEED + 800)
calibration_u_reference = np.clip(
    pit_rng.uniform(0.0, 1.0, size=N_SIMULATOR_CALIBRATION_TOTAL),
    PIT_EPS,
    1.0 - PIT_EPS,
).astype(np.float32)
calibration_positive = np.column_stack([
    calibration_mu, calibration_u0
])
calibration_negative = np.column_stack([
    calibration_mu, calibration_u_reference
])
if not np.array_equal(
    calibration_positive[:, 0], calibration_negative[:, 0]
):
    raise RuntimeError("The PIT classifier lost its matched mu design.")

paired_calibration_ids = np.arange(
    N_SIMULATOR_CALIBRATION_TOTAL, dtype=np.int64
)
calibration_ratio_ensemble = []
for member in range(RATIO_ENSEMBLE_SIZE):
    print("\n" + "=" * 76)
    print(
        f"Training PIT calibration member "
        f"{member + 1}/{RATIO_ENSEMBLE_SIZE}"
    )
    print("=" * 76)
    calibration_ratio_ensemble.append(
        train_ratio_classifier(
            calibration_positive,
            calibration_negative,
            checkpoint=(
                CALIBRATION_RATIO_MODEL_DIR
                / f"r_cal_member{member}.pt"
            ),
            model_config=CORRECTION_MODEL_CONFIG,
            training_config=CORRECTION_TRAINING_CONFIG,
            device=device,
            seed=SEED + 810 + 100 * member,
            load_if_available=LOAD_IF_AVAILABLE,
            paired_group_ids=paired_calibration_ids,
            verify_checkpoint_data=True,
        )
    )
assert all(
    pack["history"].get("split_strategy") == "paired_groups"
    for pack in calibration_ratio_ensemble
)
assert not any(
    pack["history"].get("weighted_bce", False)
    for pack in calibration_ratio_ensemble
)

fig, ax = plt.subplots(figsize=(7.0, 4.5))
for member, pack in enumerate(calibration_ratio_ensemble):
    ax.plot(
        pack.get("history", {}).get("validation", []),
        label=f"member {member}",
    )
ax.axhline(np.log(2.0), color="black", ls="--", lw=1.2,
           label=r"$\log 2$")
ax.set(xlabel="Epoch", ylabel="Validation BCE",
       title="Residual simulator correction after response matching")
ax.grid(alpha=.25)
ax.legend(ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "pit_ratio_training")
plt.show()


## 9. Normalize the residual PIT ratio and build the calibrated primitive

The PIT reference is exactly uniform, so the only remaining normalizer is the one-dimensional integral over $u\in[0,1]$:

$$
G(u\mid\mu)=
\frac{\int_0^u r_{\rm cal}(v,\mu)\,dv}
     {\int_0^1 r_{\rm cal}(v,\mu)\,dv}.
$$

The code checks logit clipping, normalization, monotonicity, and nested full/half-grid stability before the map can be used.


In [ ]:
U_GRID = np.linspace(0.0, 1.0, PIT_QUADRATURE_POINTS)
calibration_grid = conditional_ratio_grid(
    calibration_ratio_ensemble,
    MU_DENSITY_GRID,
    U_GRID,
    max_abs_log_ratio=LOG_RATIO_CLIP,
)
print(
    "raw log Z_cal(mu) quantiles:",
    np.quantile(
        calibration_grid["log_normalization"],
        [0, .01, .5, .99, 1],
    ),
)
print(
    "PIT-ratio logit range / clipped fraction:",
    calibration_grid["ratio_log_range"],
    f"{calibration_grid['ratio_clip_fraction']:.4%}",
)
if (
    np.max(np.abs(calibration_grid["cdf"][:, 0])) > 1.0e-10
    or np.max(np.abs(calibration_grid["cdf"][:, -1] - 1.0))
    > 1.0e-10
    or np.min(np.diff(calibration_grid["cdf"], axis=1)) < -1.0e-10
):
    raise RuntimeError("G is not a valid conditional CDF.")

PIT_EVALUATION_OVERFLOW_COUNTS = {"lower": 0, "upper": 0}


def evaluate_calibrated_pit(mu, t_mu):
    """Return (U0, Ucal), using certified CDF limits off-grid."""
    mu = np.asarray(mu, dtype=np.float64).reshape(-1)
    t_mu = np.asarray(t_mu, dtype=np.float64).reshape(-1)
    if (
        len(mu) != len(t_mu)
        or not np.isfinite(mu).all()
        or np.isnan(t_mu).any()
        or np.any(t_mu < 0.0)
    ):
        raise ValueError(
            "mu/t_mu must be matched; mu must be finite and t_mu "
            "must be non-negative and not NaN."
        )
    if np.any(mu <= MU_RANGE[0]) or np.any(mu >= MU_RANGE[1]):
        raise ValueError("Use the empirical rules at exact endpoints.")
    y = np.log(t_mu + T_OFFSET)
    if np.any(y < Y_MIN - 1.0e-10):
        raise RuntimeError(
            "A non-negative statistic fell materially below log(T_OFFSET)."
        )
    lower = y < Y_MIN
    upper = y > Y_MAX
    interior = ~(lower | upper)
    u0 = np.empty_like(y)
    u_cal = np.empty_like(y)
    if np.any(interior):
        u0[interior] = conditional_cdf_values(
            hybrid1_grid["cdf"], MU_DENSITY_GRID, Y_GRID,
            mu[interior], y[interior],
        )
        u_cal[interior] = conditional_cdf_values(
            calibration_grid["cdf"], MU_DENSITY_GRID, U_GRID,
            mu[interior], u0[interior],
        )
    u0[lower] = 0.0
    u_cal[lower] = 0.0
    u0[upper] = 1.0
    u_cal[upper] = 1.0
    for side, mask in (("lower", lower), ("upper", upper)):
        count = int(mask.sum())
        if count:
            PIT_EVALUATION_OVERFLOW_COUNTS[side] += count
            print(
                f"PIT evaluation mapped {count:,} {side}-support rows to "
                f"the exact CDF limit (cumulative "
                f"{PIT_EVALUATION_OVERFLOW_COUNTS[side]:,})."
            )
    return u0, u_cal


def evaluate_calibrated_statistic(mu, t_mu):
    """Map Ucal to a chi-square-looking scalar statistic."""
    _, u_cal = evaluate_calibrated_pit(mu, t_mu)
    return chi2.ppf(np.clip(u_cal, 1.0e-12, 1.0 - 1.0e-12), df=1)


_, calibration_u_cal = evaluate_calibrated_pit(
    calibration_mu, simulator_calibration_toys["t_mu"]
)

selected_mu = [0.25, 1.0, 2.0, 2.75]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
for mu_value in selected_mu:
    index = int(np.argmin(np.abs(MU_DENSITY_GRID - mu_value)))
    axes[0].plot(
        U_GRID,
        calibration_grid["density"][index],
        lw=1.8,
        label=rf"$\mu={mu_value:g}$",
    )
    axes[1].plot(
        U_GRID,
        calibration_grid["cdf"][index],
        lw=1.8,
        label=rf"$\mu={mu_value:g}$",
    )
axes[0].axhline(1.0, color="black", ls="--", lw=1.2,
                label="Uniform density")
axes[1].plot(U_GRID, U_GRID, "k--", lw=1.2,
             label="Identity / no correction")
axes[0].set(xlabel=r"response-matched PIT $u_0$", ylabel=r"$g_{\rm cal}(u_0\mid\mu)$",
            title="Learned simulator/hNDE ratio")
axes[1].set(xlabel=r"response-matched PIT $u_0$", ylabel=r"$G(u_0\mid\mu)$",
            title="Conditional calibration map")
for ax in axes:
    ax.grid(alpha=.2)
    ax.legend(fontsize=8)
fig.tight_layout()
export_exercise11_figure(fig, "pit_ratio_and_primitive")
plt.show()

bins = np.linspace(0.0, 1.0, 41)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
axes[0].hist(calibration_u0_raw, bins=bins, density=True,
             histtype="step", lw=2, color="C1")
axes[1].hist(calibration_u_cal, bins=bins, density=True,
             histtype="step", lw=2, color="C0")
for ax in axes:
    ax.axhline(1.0, color="black", ls="--", lw=1.2)
    ax.set(xlim=(0, 1), xlabel="PIT value", ylabel="Density")
    ax.grid(alpha=.2)
axes[0].set_title(r"Before calibration: $U_0=F_{\rm H}(T\mid\mu)$")
axes[1].set_title(r"After calibration: $U_{\rm cal}=G(U_0\mid\mu)$")
fig.tight_layout()
export_exercise11_figure(fig, "pit_calibration_training_closure")
plt.show()


## 10. Conditional quantiles, endpoints, and Neyman inversion

The calibrated $\gamma$ quantile follows without sampling the learned density:

$$
u_\gamma(\mu)=G^{-1}(\gamma\mid\mu),
\qquad
c_\gamma(\mu)
=\left(F_{\rm H}^{(g)}\right)^{-1}
\!\left(u_\gamma(\mu)\mid\mu\right).
$$

We compare these curves with the response-matched hNDE quantiles and with direct, coarse binned simulator quantiles. The latter use the calibration ensemble and are therefore only a construction diagnostic.

### Exact endpoints

The continuous proposal draws no point exactly at $\mu=0$ or $3$, while the compressed Poisson statistic remains formally discrete. Under the old identity response, $g(0)=0$ and the constrained MLE created a large atom at $t_0=0$. After response matching that specific atom persists only if $g(0)=0$; it must not be assumed. Nevertheless, neither continuous interpolation nor a continuous density-ratio model is the right tool for an exact endpoint that received no training mass.

We therefore generate explicit construction-template endpoint ensembles and use the finite-sample conservative order statistic

$$
k=\left\lceil\gamma(n+1)\right\rceil,
\qquad c_\gamma=T_{(k)}.
$$

Ties can make this non-randomized endpoint test conservative. Endpoint quantiles are treated piecewise and are not inserted into the smooth interior cutoff interpolant. The helper below applies calibrated PIT inversion on the open interval and empirical cutoffs at the two exact endpoints.


In [ ]:
calibration_u0_quantiles = conditional_quantiles(
    calibration_grid["cdf"], U_GRID, QUANTILE_LEVELS
)
calibrated_quantile_y = conditional_row_quantiles(
    hybrid1_grid["cdf"],
    Y_GRID,
    calibration_u0_quantiles,
)
calibrated_quantile_t = np.maximum(
    np.exp(calibrated_quantile_y) - T_OFFSET, 0.0
)

coarse_u_grid = U_GRID[::2]
coarse_calibration_density = calibration_grid["density"][:, ::2]
coarse_calibration_normalization = trapezoid(
    coarse_calibration_density, x=coarse_u_grid, axis=1
)
coarse_calibration_cdf = cumulative_trapezoid(
    coarse_calibration_density,
    x=coarse_u_grid,
    axis=1,
    initial=0.0,
) / coarse_calibration_normalization[:, None]
coarse_calibration_u0_quantiles = conditional_quantiles(
    coarse_calibration_cdf, coarse_u_grid, QUANTILE_LEVELS
)
coarse_calibrated_quantile_y = conditional_row_quantiles(
    hybrid1_grid["cdf"],
    Y_GRID,
    coarse_calibration_u0_quantiles,
)
u_resolution_shift = float(np.max(np.abs(
    coarse_calibrated_quantile_y - calibrated_quantile_y
)))
u_resolution_tolerance = max(0.02, 4.0 * np.max(np.diff(Y_GRID)))
print(
    "Full/half u-grid maximum final-quantile shift / tolerance:",
    u_resolution_shift, u_resolution_tolerance,
)
if u_resolution_shift > u_resolution_tolerance:
    raise RuntimeError(
        "The calibrated conditional quantiles are not stable when "
        "the u-grid resolution is halved. Increase "
        "PIT_QUADRATURE_POINTS."
    )

calibrated_cdf_grid = np.asarray([
    np.interp(f_h_row, U_GRID, g_row)
    for f_h_row, g_row in zip(
        hybrid1_grid["cdf"], calibration_grid["cdf"]
    )
])
composed_quantile_y = conditional_quantiles(
    calibrated_cdf_grid, Y_GRID, QUANTILE_LEVELS
)
composition_difference = np.max(np.abs(
    composed_quantile_y - calibrated_quantile_y
))
print(
    "Maximum nested/composed quantile difference in y:",
    composition_difference,
)
if composition_difference > 2.0 * np.max(np.diff(Y_GRID)):
    raise RuntimeError("Nested and composed conditional CDFs disagree.")

index_95 = int(np.flatnonzero(
    np.isclose(QUANTILE_LEVELS, .95)
)[0])
critical_hnde_grid = hybrid1_quantile_t[:, index_95]
critical_calibrated_grid = calibrated_quantile_t[:, index_95]
critical_calibrated_smooth = PchipInterpolator(
    MU_DENSITY_GRID, critical_calibrated_grid, extrapolate=False
)

anchor_hnde = {}
anchor_simulator = {}
anchor_hnde_quantiles = []
anchor_simulator_quantiles = []
for anchor_mu in ANCHOR_MUS:
    key = f"mu_{anchor_mu:g}".replace(".", "p")
    anchor_hnde[anchor_mu] = run_cached_toy_ensemble(
        cache_dir=CACHE_DIR / f"anchor_hnde_{key}_{N_ANCHOR_TOYS}",
        n_toys=N_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 900 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=HNDE_SIGNAL_PROBABILITY,
        background_probability=HNDE_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
        fit_runtime_fingerprint=TOY_FIT_RUNTIME_FINGERPRINT,
        generation_mu_transform=pseudo_truth_response,
        test_mu_transform=pseudo_truth_response,
        generation_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
        test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
    )
    anchor_simulator[anchor_mu] = run_cached_toy_ensemble(
        cache_dir=(
            PIT_CACHE_DIR
            / f"anchor_simulator_construction_{key}_{N_ANCHOR_TOYS}"
        ),
        n_toys=N_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 910 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=SIM_CALIBRATION_SIGNAL_PROBABILITY,
        background_probability=SIM_CALIBRATION_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
        fit_runtime_fingerprint=TOY_FIT_RUNTIME_FINGERPRINT,
        test_mu_transform=pseudo_truth_response,
        test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
    )
    anchor_hnde_quantiles.append(conservative_empirical_quantile(
        anchor_hnde[anchor_mu]["t_mu"], QUANTILE_LEVELS
    ))
    anchor_simulator_quantiles.append(conservative_empirical_quantile(
        anchor_simulator[anchor_mu]["t_mu"], QUANTILE_LEVELS
    ))
anchor_hnde_quantiles = np.asarray(anchor_hnde_quantiles)
anchor_simulator_quantiles = np.asarray(anchor_simulator_quantiles)

def neyman_accepts_95(mu, t_mu):
    """Apply the endpoint-aware rule to response-corrected t_mu^(g)."""
    mu, t_mu = np.broadcast_arrays(
        np.asarray(mu, dtype=np.float64),
        np.asarray(t_mu, dtype=np.float64),
    )
    output_shape = mu.shape
    mu = mu.reshape(-1)
    t_mu = t_mu.reshape(-1)
    if np.any(t_mu < 0.0) or not np.isfinite(t_mu).all():
        raise ValueError("t_mu must be finite and non-negative.")
    if (
        not np.isfinite(mu).all()
        or np.any(mu < MU_RANGE[0])
        or np.any(mu > MU_RANGE[1])
    ):
        raise ValueError("mu lies outside the calibrated parameter range.")
    accepted = np.empty(len(mu), dtype=bool)
    assigned = np.zeros(len(mu), dtype=bool)
    interior = (mu > MU_RANGE[0]) & (mu < MU_RANGE[1])
    if np.any(interior):
        _, u_cal = evaluate_calibrated_pit(mu[interior], t_mu[interior])
        accepted[interior] = u_cal <= .95
        assigned[interior] = True
    for anchor_index, anchor_mu in enumerate(ANCHOR_MUS):
        endpoint = mu == anchor_mu
        accepted[endpoint] = (
            t_mu[endpoint]
            <= anchor_simulator_quantiles[anchor_index, index_95]
        )
        assigned[endpoint] = True
    if not np.all(assigned):
        raise ValueError("mu lies outside the calibrated parameter range.")
    return accepted.reshape(output_shape)

calibration_quantile_edges = np.linspace(*MU_RANGE, 21)
calibration_quantile_centers = 0.5 * (
    calibration_quantile_edges[:-1] + calibration_quantile_edges[1:]
)
calibration_bin_index = np.clip(
    np.digitize(calibration_mu, calibration_quantile_edges) - 1,
    0,
    len(calibration_quantile_centers) - 1,
)
direct_calibration_q95 = np.asarray([
    conservative_empirical_quantile(
        simulator_calibration_toys["t_mu"][calibration_bin_index == i],
        .95,
    )
    for i in range(len(calibration_quantile_centers))
])

np.savez_compressed(
    PIT_CACHE_DIR / "conditional_pit_calibration.npz",
    mu=MU_DENSITY_GRID,
    levels=QUANTILE_LEVELS,
    hnde=hybrid1_quantile_t,
    simulator_calibrated=calibrated_quantile_t,
    endpoint_mu=ANCHOR_MUS,
    endpoint_hnde=anchor_hnde_quantiles,
    endpoint_simulator=anchor_simulator_quantiles,
    u_grid=U_GRID,
    calibration_cdf=calibration_grid["cdf"],
    response_mu=RESPONSE_MU_GRID,
    response_g=response_grid,
)

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))
colors = plt.cm.viridis(np.linspace(.12, .9, len(QUANTILE_LEVELS)))
interior_grid = slice(1, -1)
for level, color, before, after in zip(
    QUANTILE_LEVELS,
    colors,
    hybrid1_quantile_t.T,
    calibrated_quantile_t.T,
):
    linewidth = 2.8 if np.isclose(level, .95) else 1.25
    axes[0].plot(
        MU_DENSITY_GRID[interior_grid], before[interior_grid],
        color=color, ls="--", lw=linewidth,
        label=(rf"{level:.0%} response-matched hNDE"
               if np.isclose(level, .95)
               else rf"{level:.0%}"),
    )
    axes[0].plot(
        MU_DENSITY_GRID[interior_grid], after[interior_grid],
        color=color, lw=linewidth,
        label=(rf"{level:.0%} calibrated"
               if np.isclose(level, .95) else None),
    )
    axes[1].plot(
        MU_DENSITY_GRID[interior_grid],
        (after - before)[interior_grid],
        color=color, lw=linewidth, label=rf"{level:.0%}",
    )
axes[0].scatter(
    ANCHOR_MUS,
    anchor_simulator_quantiles[:, index_95],
    marker="s", s=42, color="C3", zorder=5,
    label="empirical endpoint 95%",
)
axes[0].scatter(
    calibration_quantile_centers,
    direct_calibration_q95,
    marker="o", s=18, facecolors="none", edgecolors="0.25",
    label="binned simulator 95%",
)
axes[0].set(
    xlabel=r"$\mu$", ylabel=r"conditional quantile of $t_\mu$",
    title="Response-matched baseline and residual PIT calibration",
)
axes[1].axhline(0.0, color="black", ls=":", lw=1)
axes[1].set(
    xlabel=r"$\mu$", ylabel="calibrated − hNDE quantile",
    title="Calibration displacement",
)
for ax in axes:
    ax.grid(alpha=.2)
    ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "conditional_pit_quantile_corrections")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0),
                         sharex=True, sharey=True)
t_grid = np.maximum(np.exp(Y_GRID) - T_OFFSET, 0.0)
for ax, mu_value in zip(axes.flat, [0.25, 1.0, 2.0, 2.75]):
    index = int(np.argmin(np.abs(MU_DENSITY_GRID - mu_value)))
    ax.plot(t_grid, hybrid1_grid["cdf"][index], ls="--", lw=2,
            label="response-matched hNDE")
    ax.plot(t_grid, calibrated_cdf_grid[index], lw=2,
            label="PIT-ratio calibrated")
    ax.axhline(.95, color="0.4", ls=":", lw=1)
    ax.set_xlim(
        0, max(8.0, float(critical_calibrated_grid[index]) * 1.5)
    )
    ax.set_title(rf"$\mu={mu_value:g}$")
    ax.grid(alpha=.2)
for ax in axes[-1]:
    ax.set_xlabel(r"$t_\mu$")
for ax in axes[:, 0]:
    ax.set_ylabel("Conditional CDF")
axes[0, 0].legend()
fig.tight_layout()
export_exercise11_figure(fig, "conditional_pit_cdf_primitives")
plt.show()


## 11. Same-template internal audit: isolate calibration error

Before opening the independently seeded 64M audit law, generate a fresh toy ensemble from the same fixed 64M construction template used by $g$ and by the PIT ratio. These toys have new random seeds and were not used for training. This control answers a narrow question: **does the learned conditional calibration reproduce its own declared construction law?**

It does not test event-level simulator transfer or finite-template uncertainty. Keeping this control separate is useful: a failure here is calibration-model error, whereas a discrepancy that appears only after the sealed audit is opened is evidence of sensitivity to the finite construction reservoir.


In [ ]:
simulator_internal_audit_toys = run_cached_toy_ensemble(
    cache_dir=(
        PIT_CACHE_DIR
        / f"simulator_construction_internal_audit_"
          f"{N_SIMULATOR_INTERNAL_AUDIT_TOYS}"
    ),
    n_toys=N_SIMULATOR_INTERNAL_AUDIT_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 1000,
    mu_range=MU_RANGE,
    signal_probability=SIM_CALIBRATION_SIGNAL_PROBABILITY,
    background_probability=SIM_CALIBRATION_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
    fit_runtime_fingerprint=TOY_FIT_RUNTIME_FINGERPRINT,
    test_mu_transform=pseudo_truth_response,
    test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
)
internal_mu = simulator_internal_audit_toys["mu"].astype(np.float64)
internal_t = simulator_internal_audit_toys["t_mu"].astype(np.float64)
if not np.allclose(
    simulator_internal_audit_toys["generation_mu"], internal_mu,
    rtol=0.0, atol=2.0e-6,
) or not np.allclose(
    simulator_internal_audit_toys["test_mu"],
    pseudo_truth_response(internal_mu),
    rtol=0.0, atol=2.0e-6,
):
    raise RuntimeError("The internal-audit cache has stale coordinates.")
internal_u0, internal_u_cal = evaluate_calibrated_pit(
    internal_mu, internal_t
)
internal_covered_hnde = internal_u0 <= .95
internal_covered_calibrated = internal_u_cal <= .95
if not np.array_equal(
    neyman_accepts_95(internal_mu, internal_t),
    internal_covered_calibrated,
):
    raise RuntimeError(
        "Endpoint-aware decisions disagree in the open-interval internal audit."
    )
coverage_edges = np.linspace(*MU_RANGE, 21)
binned_internal = binned_coverage(
    internal_mu, internal_covered_calibrated, edges=coverage_edges
)
print(
    "Internal same-template response-baseline coverage:",
    f"{internal_covered_hnde.mean():.4%}",
)
print(
    "Internal same-template PIT-calibrated coverage:",
    f"{internal_covered_calibrated.mean():.4%}",
)
print(
    "Internal calibrated binned range:",
    f"[{binned_internal['coverage'].min():.4%}, "
    f"{binned_internal['coverage'].max():.4%}]",
)


## 12. Finite-construction-template bootstrap diagnostic

The calibration reservoir contains a finite number of simulator events. To measure how much the *fixed* construction changes when that empirical law fluctuates, resample the signal and background compressed bin counts multinomially, generate toys under each bootstrap template, and apply the already frozen response map and calibrated pivot.

This is a sensitivity diagnostic, not another independent coverage audit. Every replicate is derived from the same construction events. Its spread combines template sensitivity with finite toy noise; Wilson intervals show the latter for each replicate. Recalibrating separately inside every bootstrap would answer a different hierarchical question and would not test the robustness of this fixed construction.


In [ ]:
def multinomial_template_bootstrap(counts, rng):
    counts = np.asarray(counts, dtype=np.int64)
    if np.any(counts < 0) or counts.sum() <= 0:
        raise ValueError("Bootstrap template counts must be non-negative.")
    draw = rng.multinomial(
        int(counts.sum()), counts / counts.sum()
    ).astype(np.float64)
    draw += 1.0e-15
    return draw / draw.sum()


template_bootstrap_rows = []
template_bootstrap_binned = []
bootstrap_edges = np.linspace(*MU_RANGE, 11)
for replicate in range(N_TEMPLATE_BOOTSTRAPS):
    rng = np.random.default_rng(
        np.random.SeedSequence([SEED, 1200, replicate])
    )
    bootstrap_signal_probability = multinomial_template_bootstrap(
        SIM_CALIBRATION_SIGNAL_COUNTS, rng
    )
    bootstrap_background_probability = multinomial_template_bootstrap(
        SIM_CALIBRATION_BACKGROUND_COUNTS, rng
    )
    bootstrap_toys = run_cached_toy_ensemble(
        cache_dir=(
            PIT_CACHE_DIR
            / f"construction_template_bootstrap_{replicate:03d}_"
              f"{N_TOYS_PER_TEMPLATE_BOOTSTRAP}"
        ),
        n_toys=N_TOYS_PER_TEMPLATE_BOOTSTRAP,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 1300 + replicate,
        mu_range=MU_RANGE,
        signal_probability=bootstrap_signal_probability,
        background_probability=bootstrap_background_probability,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
        fit_runtime_fingerprint=TOY_FIT_RUNTIME_FINGERPRINT,
        test_mu_transform=pseudo_truth_response,
        test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
    )
    bootstrap_covered = neyman_accepts_95(
        bootstrap_toys["mu"], bootstrap_toys["t_mu"]
    )
    successes = int(bootstrap_covered.sum())
    lower, upper = wilson_interval(successes, len(bootstrap_covered))
    template_bootstrap_rows.append({
        "replicate": replicate,
        "coverage": float(bootstrap_covered.mean()),
        "toy_wilson_lower": float(lower),
        "toy_wilson_upper": float(upper),
    })
    template_bootstrap_binned.append(
        binned_coverage(
            bootstrap_toys["mu"],
            bootstrap_covered,
            edges=bootstrap_edges,
        )["coverage"]
    )

template_bootstrap_table = pd.DataFrame(template_bootstrap_rows)
template_bootstrap_binned = np.asarray(template_bootstrap_binned)
bootstrap_global_interval = np.quantile(
    template_bootstrap_table["coverage"], [.025, .5, .975]
)
bootstrap_binned_interval = np.quantile(
    template_bootstrap_binned, [.025, .5, .975], axis=0
)
display(
    template_bootstrap_table.describe().T.style.format(precision=6)
)
print(
    "Construction-template bootstrap global 2.5/50/97.5% quantiles:",
    bootstrap_global_interval,
)
penultimate = template_convergence_table[
    (template_convergence_table["source"] == "fresh nested construction")
    & (template_convergence_table["selected_per_process"] == int(TEMPLATE_SELECTED_CHECKPOINTS[-2]))
].iloc[0]
print(
    f"Penultimate-prefix ({int(TEMPLATE_SELECTED_CHECKPOINTS[-2]):,}) "
    f"L1 to the {FINAL_TEMPLATE_EVENTS:,} construction: "
    f"signal={penultimate['signal_L1_to_64M']:.6f}, "
    f"background={penultimate['background_L1_to_64M']:.6f}."
)


## 13. Open the new 64M audit and perform the primary LF2I coverage test

Everything that defines the confidence procedure is now frozen: the 64M construction law, $g_{64\mathrm M}$, response-matched hNDE reference, first ratio, PIT ratio, numerical normalizations, endpoint rules, and interpolated cutoffs. Only now are the independently seeded audit $q$ and feature histograms loaded.

The audit first reports construction-to-audit differences in four complementary coordinates:

- PRESEL acceptance;
- selected-feature histograms;
- raw $q$-template $L^1$;
- the likelihood-score direction, through $|S_A(g_N;\mu)|/\sqrt{I_A(g_N;\mu)}$ and $g_N(\mu)-g_A(\mu)$.

The 500,000-toy continuous audit that follows is the primary external coverage result. The later prefix-transfer study is explicitly post-unsealing and diagnostic; it must not be used to retune this already frozen construction.


In [ ]:
AUDIT_STREAM_RESULTS = {}
for process in ("signal", "background"):
    AUDIT_STREAM_RESULTS[process] = load_streamed_simulator_template(
        cache_path=SEALED_AUDIT_TEMPLATE_PATHS[process],
        process=process,
        selected_checkpoints=TEMPLATE_SELECTED_CHECKPOINTS,
        batch_size=FRESH_SIMULATOR_BATCH_SIZE,
        seed=FRESH_AUDIT_SEED[process],
        feature_names=FEATURES,
        feature_edges=FEATURE_DIAGNOSTIC_EDGES,
        log_q_edges=LOG_Q_EDGES,
        presel_ratio_cut=PRESEL_RATIO_CUT,
        recipe_fingerprint=f"{STREAM_RECIPE_FINGERPRINT}:audit",
        expose_counts=True,
    )
    if (
        AUDIT_STREAM_RESULTS[process]["fingerprint"]
        != AUDIT_STREAM_SUMMARIES[process]["fingerprint"]
    ):
        raise RuntimeError(
            f"The opened {process} audit differs from its pre-freeze "
            "cryptographic commitment."
        )

SIM_AUDIT_SIGNAL_COUNTS_BY_CHECKPOINT = np.asarray(
    AUDIT_STREAM_RESULTS["signal"]["q_counts"], dtype=np.int64
)
SIM_AUDIT_BACKGROUND_COUNTS_BY_CHECKPOINT = np.asarray(
    AUDIT_STREAM_RESULTS["background"]["q_counts"], dtype=np.int64
)
SIM_AUDIT_SIGNAL_FEATURE_COUNTS_BY_CHECKPOINT = np.asarray(
    AUDIT_STREAM_RESULTS["signal"]["feature_counts"], dtype=np.int64
)
SIM_AUDIT_BACKGROUND_FEATURE_COUNTS_BY_CHECKPOINT = np.asarray(
    AUDIT_STREAM_RESULTS["background"]["feature_counts"], dtype=np.int64
)
SIM_AUDIT_SIGNAL_COUNTS = SIM_AUDIT_SIGNAL_COUNTS_BY_CHECKPOINT[-1]
SIM_AUDIT_BACKGROUND_COUNTS = SIM_AUDIT_BACKGROUND_COUNTS_BY_CHECKPOINT[-1]
SIM_AUDIT_SIGNAL_PROBABILITY = SIM_AUDIT_SIGNAL_COUNTS / FINAL_TEMPLATE_EVENTS
SIM_AUDIT_BACKGROUND_PROBABILITY = SIM_AUDIT_BACKGROUND_COUNTS / FINAL_TEMPLATE_EVENTS
SIM_AUDIT_EVENTS = {
    process: int(AUDIT_STREAM_RESULTS[process]["selected_events"])
    for process in ("signal", "background")
}
if SIM_AUDIT_EVENTS != {
    "signal": FINAL_TEMPLATE_EVENTS, "background": FINAL_TEMPLATE_EVENTS
}:
    raise RuntimeError("The newly opened audit does not have the registered size.")

audit_signal_l1 = float(np.sum(np.abs(
    SIM_AUDIT_SIGNAL_PROBABILITY - SIM_CALIBRATION_SIGNAL_PROBABILITY
)))
audit_background_l1 = float(np.sum(np.abs(
    SIM_AUDIT_BACKGROUND_PROBABILITY - SIM_CALIBRATION_BACKGROUND_PROBABILITY
)))
audit_response_grid = template_response_grid(
    SIM_AUDIT_SIGNAL_PROBABILITY, SIM_AUDIT_BACKGROUND_PROBABILITY
)

audit_score_rows = []
for checkpoint_index, selected_events in enumerate(TEMPLATE_SELECTED_CHECKPOINTS):
    standardized = standardized_score_drift(
        PREFIX_RESPONSE_GRIDS[checkpoint_index],
        SIM_AUDIT_SIGNAL_PROBABILITY,
        SIM_AUDIT_BACKGROUND_PROBABILITY,
    )
    audit_score_rows.append({
        "construction_prefix": int(selected_events),
        "max_abs_gN_minus_g_audit": float(np.max(np.abs(
            PREFIX_RESPONSE_GRIDS[checkpoint_index] - audit_response_grid
        ))),
        "max_standardized_audit_score": float(np.max(standardized)),
    })
audit_score_convergence_table = pd.DataFrame(audit_score_rows)

audit_feature_rows = []
for process, construction_counts, audit_counts in (
    (
        "signal", SIM_CALIBRATION_SIGNAL_FEATURE_COUNTS_BY_CHECKPOINT[-1],
        SIM_AUDIT_SIGNAL_FEATURE_COUNTS_BY_CHECKPOINT[-1],
    ),
    (
        "background", SIM_CALIBRATION_BACKGROUND_FEATURE_COUNTS_BY_CHECKPOINT[-1],
        SIM_AUDIT_BACKGROUND_FEATURE_COUNTS_BY_CHECKPOINT[-1],
    ),
):
    for feature_index, feature in enumerate(FEATURES):
        audit_feature_rows.append({
            "process": process,
            "feature": feature,
            "construction_audit_L1": float(np.sum(np.abs(
                construction_counts[feature_index] / FINAL_TEMPLATE_EVENTS
                - audit_counts[feature_index] / FINAL_TEMPLATE_EVENTS
            ))),
        })
audit_feature_closure_table = pd.DataFrame(audit_feature_rows)

print("Opened independent 64M audit:", SIM_AUDIT_EVENTS)
print(
    "Construction/audit q-template L1 differences:",
    f"signal={audit_signal_l1:.6f}, background={audit_background_l1:.6f}",
)
display(audit_score_convergence_table.style.format({
    "construction_prefix": "{:,.0f}",
    "max_abs_gN_minus_g_audit": "{:.6f}",
    "max_standardized_audit_score": "{:.4f}",
}).hide(axis="index"))
display(audit_feature_closure_table.style.format(precision=6).hide(axis="index"))
display(pd.DataFrame([
    {
        "process": process,
        "construction_acceptance": CONSTRUCTION_STREAM_RESULTS[process]["acceptance"],
        "audit_acceptance": AUDIT_STREAM_RESULTS[process]["acceptance"],
        "relative_difference": (
            AUDIT_STREAM_RESULTS[process]["acceptance"]
            / CONSTRUCTION_STREAM_RESULTS[process]["acceptance"] - 1.0
        ),
    }
    for process in ("signal", "background")
]).style.format({
    "construction_acceptance": "{:.6%}",
    "audit_acceptance": "{:.6%}",
    "relative_difference": "{:+.4%}",
}).hide(axis="index"))

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8))
for checkpoint_index, selected_events in enumerate(TEMPLATE_SELECTED_CHECKPOINTS):
    axes[0].plot(
        RESPONSE_MU_GRID,
        PREFIX_RESPONSE_GRIDS[checkpoint_index] - audit_response_grid,
        lw=1.7, label=f"{int(selected_events):,}",
    )
axes[0].axhline(0.0, color="0.5", lw=1.0)
axes[0].set(
    xlabel=r"physical truth $\mu$",
    ylabel=r"$g_N(\mu)-g_A(\mu)$",
    title="Construction response versus new audit response",
)
axes[0].grid(alpha=.2)
axes[0].legend(fontsize=7, ncol=2)
axes[1].plot(
    audit_score_convergence_table["construction_prefix"],
    audit_score_convergence_table["max_standardized_audit_score"],
    marker="o", lw=2.0,
)
axes[1].set_xscale("log", base=4)
axes[1].set(
    xlabel="selected construction events per process",
    ylabel=r"$\max_\mu |S_A|/\sqrt{I_A}$",
    title="Score-direction transfer convergence",
)
axes[1].grid(alpha=.2)
fig.tight_layout()
export_exercise11_figure(fig, "audit_response_and_score_transfer")
plt.show()

simulator_audit_toys = run_cached_toy_ensemble(
    cache_dir=(
        PIT_CACHE_DIR
        / f"simulator_new64m_audit_{N_SIMULATOR_AUDIT_TOYS}"
    ),
    n_toys=N_SIMULATOR_AUDIT_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 1_900,
    mu_range=MU_RANGE,
    signal_probability=SIM_AUDIT_SIGNAL_PROBABILITY,
    background_probability=SIM_AUDIT_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
    fit_runtime_fingerprint=TOY_FIT_RUNTIME_FINGERPRINT,
    test_mu_transform=pseudo_truth_response,
    test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
)
audit_mu = simulator_audit_toys["mu"].astype(np.float64)
audit_t = simulator_audit_toys["t_mu"].astype(np.float64)
if not np.allclose(
    simulator_audit_toys["generation_mu"], audit_mu,
    rtol=0.0, atol=2.0e-6,
) or not np.allclose(
    simulator_audit_toys["test_mu"], pseudo_truth_response(audit_mu),
    rtol=0.0, atol=2.0e-6,
):
    raise RuntimeError("The new-audit toy cache has stale coordinates.")

audit_u0, audit_u_cal = evaluate_calibrated_pit(audit_mu, audit_t)
covered_hnde = audit_u0 <= .95
covered_calibrated = audit_u_cal <= .95
if not np.array_equal(neyman_accepts_95(audit_mu, audit_t), covered_calibrated):
    raise RuntimeError("The endpoint-aware helper disagrees with the interior PIT rule.")

audit_critical_calibrated = critical_calibrated_smooth(audit_mu)
covered_by_cutoff = audit_t <= audit_critical_calibrated
cutoff_disagreement = float(np.mean(covered_by_cutoff != covered_calibrated))
print(f"Pivot/cutoff decision disagreement: {cutoff_disagreement:.4%}")
if cutoff_disagreement > 2.0e-3:
    print(
        "WARNING: the auxiliary interpolated cutoff disagrees with more than "
        "0.2% of direct pivot decisions. Refine the mu/y/u grids before using "
        "the smooth cutoff as a numerical substitute."
    )

audit_t_cal = evaluate_calibrated_statistic(audit_mu, audit_t)
print("New-64M-audit response-baseline coverage:", f"{covered_hnde.mean():.4%}")
print("New-64M-audit PIT-calibrated coverage:", f"{covered_calibrated.mean():.4%}")
print(
    "Equivalent chi-square decision coverage:",
    f"{np.mean(audit_t_cal <= chi2.ppf(.95, df=1)):.4%}",
)

coverage_auditor = train_coverage_auditor(
    audit_mu,
    covered_calibrated,
    checkpoint=PIT_MODEL_DIR / "coverage_auditor_new64m_95cl.pt",
    model_config=AUDITOR_MODEL_CONFIG,
    training_config=AUDITOR_TRAINING_CONFIG,
    device=device,
    seed=SEED + 1_910,
    load_if_available=LOAD_IF_AVAILABLE,
)
auditor_grid = coverage_auditor_probability(coverage_auditor, MU_DENSITY_GRID)
observed_coverage = float(covered_calibrated.mean())
fixed_nominal_95_bce = -float(np.mean(
    covered_calibrated * np.log(.95)
    + (~covered_calibrated) * np.log(.05)
))
clipped_observed_coverage = np.clip(observed_coverage, 1.0e-12, 1.0 - 1.0e-12)
best_constant_bce = -(
    observed_coverage * np.log(clipped_observed_coverage)
    + (1.0 - observed_coverage) * np.log(1.0 - clipped_observed_coverage)
)
auditor_validation_history = coverage_auditor.get("history", {}).get(
    "validation", []
)
auditor_best_validation_bce = (
    float(np.min(auditor_validation_history))
    if len(auditor_validation_history) else np.nan
)
print(f"Observed-label BCE of fixed p=0.95 predictor: {fixed_nominal_95_bce:.6f}")
print(f"Full-sample best-constant BCE: {best_constant_bce:.6f}")
print(f"Auditor best validation BCE: {auditor_best_validation_bce:.6f}")
print(
    "New-64M-audit predicted-coverage range:",
    f"[{auditor_grid.min():.4%}, {auditor_grid.max():.4%}]",
)


## 14. Post-unsealing fixed-$\mu$ coverage transfer versus template size

The full neural calibration is trained only once, at the registered 64M construction size. To test whether the finite-template failure itself converges away, this post-unsealing diagnostic builds a simpler exact procedure at each nested prefix:

1. compute that prefix's response $g_N(\mu)$;
2. at six fixed $\mu$ values, obtain a conservative empirical 95% critical value from toys generated by the same prefix law;
3. apply the critical value to common toys from the independent 64M audit law, evaluated with the same $g_N(\mu)$.

This directly measures conditional coverage transfer without training five separate spline/PIT chains. The full run uses 50,000 construction and 50,000 audit toys at every $(N,\mu)$ point. The audit toy seed at each $\mu$ is shared across all $N$, giving common random numbers and a cleaner convergence comparison. Reported Wilson intervals quantify audit-toy Monte Carlo error conditional on the estimated construction cutoff; they do not include its order-statistic uncertainty. The study occurs after the primary audit has opened and is therefore diagnostic only: it cannot justify tuning the already reported 64M method.

In [ ]:
prefix_transfer_rows = []
for checkpoint_index, selected_events in enumerate(TEMPLATE_SELECTED_CHECKPOINTS):
    prefix_signal_probability = SIM_CALIBRATION_SIGNAL_PROBABILITY_BY_CHECKPOINT[
        checkpoint_index
    ]
    prefix_background_probability = SIM_CALIBRATION_BACKGROUND_PROBABILITY_BY_CHECKPOINT[
        checkpoint_index
    ]
    response_interpolator_N = PchipInterpolator(
        RESPONSE_MU_GRID,
        PREFIX_RESPONSE_GRIDS[checkpoint_index],
        extrapolate=False,
    )

    def response_N(mu, interpolator=response_interpolator_N):
        values = np.asarray(mu, dtype=np.float64)
        return np.asarray(interpolator(values), dtype=np.float64)

    response_fingerprint_N = PREFIX_RESPONSE_FINGERPRINTS[checkpoint_index]
    for mu_index, fixed_mu in enumerate(PREFIX_TRANSFER_MUS):
        key = f"mu_{fixed_mu:g}".replace(".", "p")
        construction_toys_N = run_cached_toy_ensemble(
            cache_dir=(
                PIT_CACHE_DIR / "prefix_transfer"
                / f"construction_N{int(selected_events)}_{key}_{N_PREFIX_TRANSFER_TOYS}"
            ),
            n_toys=N_PREFIX_TRANSFER_TOYS,
            batch_size=TOY_BATCH_SIZE,
            seed=SEED + 2_000 + mu_index,
            mu_range=MU_RANGE,
            fixed_mu=float(fixed_mu),
            signal_probability=prefix_signal_probability,
            background_probability=prefix_background_probability,
            lam_signal=LAM_SIG,
            lam_background=LAM_BKG,
            likelihood_q=COMPRESSED_Q,
            fit_batch=fit_toy_batch_numpy,
            fit_fingerprint=TOY_FIT_FINGERPRINT,
            fit_runtime_fingerprint=TOY_FIT_RUNTIME_FINGERPRINT,
            test_mu_transform=response_N,
            test_mu_fingerprint=response_fingerprint_N,
        )
        critical_N = float(conservative_empirical_quantile(
            construction_toys_N["t_mu"], .95
        ))
        audit_toys_N = run_cached_toy_ensemble(
            cache_dir=(
                PIT_CACHE_DIR / "prefix_transfer"
                / f"audit_N{int(selected_events)}_{key}_{N_PREFIX_TRANSFER_TOYS}"
            ),
            n_toys=N_PREFIX_TRANSFER_TOYS,
            batch_size=TOY_BATCH_SIZE,
            # The same seed and audit probabilities at fixed mu yield common
            # audit counts across every construction prefix N.
            seed=SEED + 2_100 + mu_index,
            mu_range=MU_RANGE,
            fixed_mu=float(fixed_mu),
            signal_probability=SIM_AUDIT_SIGNAL_PROBABILITY,
            background_probability=SIM_AUDIT_BACKGROUND_PROBABILITY,
            lam_signal=LAM_SIG,
            lam_background=LAM_BKG,
            likelihood_q=COMPRESSED_Q,
            fit_batch=fit_toy_batch_numpy,
            fit_fingerprint=TOY_FIT_FINGERPRINT,
            fit_runtime_fingerprint=TOY_FIT_RUNTIME_FINGERPRINT,
            test_mu_transform=response_N,
            test_mu_fingerprint=response_fingerprint_N,
        )
        same_template_coverage = float(np.mean(
            construction_toys_N["t_mu"] <= critical_N
        ))
        audit_covered_N = audit_toys_N["t_mu"] <= critical_N
        successes = int(audit_covered_N.sum())
        lower, upper = wilson_interval(successes, len(audit_covered_N))
        prefix_transfer_rows.append({
            "selected_per_process": int(selected_events),
            "mu": float(fixed_mu),
            "critical_95": critical_N,
            "same_template_coverage": same_template_coverage,
            "audit_transfer_coverage": float(audit_covered_N.mean()),
            "audit_mc_lower_given_cutoff": float(lower),
            "audit_mc_upper_given_cutoff": float(upper),
        })

prefix_transfer_table = pd.DataFrame(prefix_transfer_rows)
display(prefix_transfer_table.style.format({
    "selected_per_process": "{:,.0f}",
    "mu": "{:.2f}",
    "critical_95": "{:.5f}",
    "same_template_coverage": "{:.4%}",
    "audit_transfer_coverage": "{:.4%}",
    "audit_mc_lower_given_cutoff": "{:.4%}",
    "audit_mc_upper_given_cutoff": "{:.4%}",
}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8.8, 5.6))
ax.axhline(.95, color="black", ls="--", lw=1.3, label="nominal 95%")
for fixed_mu in PREFIX_TRANSFER_MUS:
    subset = prefix_transfer_table[prefix_transfer_table["mu"] == fixed_mu]
    ax.plot(
        subset["selected_per_process"], subset["audit_transfer_coverage"],
        marker="o", lw=1.7, label=rf"$\mu={fixed_mu:g}$",
    )
ax.set_xscale("log", base=4)
ax.set(
    xlabel="selected construction events per process",
    ylabel="coverage on common independent 64M-audit toys",
    title=r"Fixed-$\mu$ empirical coverage transfer versus template size",
)
ax.grid(alpha=.2)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "fixed_mu_template_size_coverage_transfer")
plt.show()

In [ ]:
binned_before = binned_coverage(
    audit_mu, covered_hnde, edges=coverage_edges
)

binned_after = binned_coverage(
    audit_mu, covered_calibrated, edges=coverage_edges
)

anchor_audit_rows = []
for anchor_index, anchor_mu in enumerate(ANCHOR_MUS):
    key = f"mu_{anchor_mu:g}".replace(".", "p")
    anchor_audit = run_cached_toy_ensemble(
        cache_dir=(
            PIT_CACHE_DIR
            / f"anchor_simulator_new64m_audit_"
              f"{key}_{N_AUDIT_ANCHOR_TOYS}"
        ),
        n_toys=N_AUDIT_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 2200 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=SIM_AUDIT_SIGNAL_PROBABILITY,
        background_probability=SIM_AUDIT_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
        fit_runtime_fingerprint=TOY_FIT_RUNTIME_FINGERPRINT,
        test_mu_transform=pseudo_truth_response,
        test_mu_fingerprint=RESPONSE_MAP_FINGERPRINT,
    )
    endpoint_critical = float(
        anchor_simulator_quantiles[anchor_index, index_95]
    )
    endpoint_covered = neyman_accepts_95(
        np.full(N_AUDIT_ANCHOR_TOYS, anchor_mu),
        anchor_audit["t_mu"],
    )
    successes = int(endpoint_covered.sum())
    lower, upper = wilson_interval(successes, len(endpoint_covered))
    anchor_audit_rows.append({
        "mu": anchor_mu,
        "construction_critical_value": endpoint_critical,
        "new_64M_audit_coverage": float(endpoint_covered.mean()),
        "wilson_lower": float(lower),
        "wilson_upper": float(upper),
        "zero_mass": float(np.mean(
            anchor_audit["t_mu"] <= 1.0e-12
        )),
    })
anchor_audit_table = pd.DataFrame(anchor_audit_rows)
display(anchor_audit_table.style.format(precision=5).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8.8, 5.6))
ax.axhline(.95, color="black", ls="--", lw=1.5,
           label="Nominal 95%")
ax.plot(MU_DENSITY_GRID, auditor_grid, color="C3", lw=2.3,
        label="new 64M-audit BCE auditor")
ax.errorbar(
    binned_before["center"], binned_before["coverage"],
    yerr=[
        binned_before["coverage"] - binned_before["lower"],
        binned_before["upper"] - binned_before["coverage"],
    ],
    fmt="o", ms=4, color="0.5", alpha=.75,
    label="response baseline, new 64M audit",
)
ax.errorbar(
    binned_internal["center"], binned_internal["coverage"],
    yerr=[
        binned_internal["coverage"] - binned_internal["lower"],
        binned_internal["upper"] - binned_internal["coverage"],
    ],
    fmt="D", ms=4, color="C1", alpha=.8,
    label="calibrated, same-template control",
)
ax.errorbar(
    binned_after["center"], binned_after["coverage"],
    yerr=[
        binned_after["coverage"] - binned_after["lower"],
        binned_after["upper"] - binned_after["coverage"],
    ],
    fmt="o", ms=5, color="C0",
    label="calibrated, new 64M audit",
)
ax.errorbar(
    anchor_audit_table["mu"],
    anchor_audit_table["new_64M_audit_coverage"],
    yerr=[
        anchor_audit_table["new_64M_audit_coverage"]
        - anchor_audit_table["wilson_lower"],
        anchor_audit_table["wilson_upper"]
        - anchor_audit_table["new_64M_audit_coverage"],
    ],
    fmt="s", ms=6, color="C2",
    label="new 64M-audit endpoint anchors",
)
ax.set(
    xlim=MU_RANGE,
    xlabel=r"physical truth $\mu$",
    ylabel="Conditional coverage",
    title="Independent 64M-event-template LF2I coverage audit",
)
ax.set_ylim(min(.93, binned_before["lower"].min() - .005), 1.005)
ax.grid(alpha=.2)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "lf2i_response_pit_new64m_audit")
plt.show()

coverage_summary = pd.DataFrame({
    "method": [
        "response baseline — new 64M audit",
        "PIT calibrated — same-template control",
        "PIT calibrated — new 64M audit",
    ],
    "global_coverage": [
        covered_hnde.mean(),
        internal_covered_calibrated.mean(),
        covered_calibrated.mean(),
    ],
    "minimum_binned_coverage": [
        binned_before["coverage"].min(),
        binned_internal["coverage"].min(),
        binned_after["coverage"].min(),
    ],
    "maximum_binned_coverage": [
        binned_before["coverage"].max(),
        binned_internal["coverage"].max(),
        binned_after["coverage"].max(),
    ],
})
display(coverage_summary.style.format(precision=5).hide(axis="index"))
print(
    "Construction-template bootstrap global 95% sensitivity interval:",
    bootstrap_global_interval[[0, 2]],
)


## 15. Invert the pivot for pseudo-observed datasets

Coverage controls the size of the test, but it does not determine the usefulness of the resulting confidence set. We now draw several pseudo-observed datasets from the newly opened independent 64M audit law and invert both:

- the response-matched hNDE reference rule before the residual PIT correction;
- the fully simulator-calibrated rule.

The scan preserves disconnected confidence-set components rather than silently replacing them with their convex hull. Total accepted length is reported for each dataset. This comparison becomes substantively meaningful only if the preceding independent audit supports the claimed 95% coverage; otherwise it remains a numerical demonstration of inversion.


In [ ]:
OBSERVED_TRUTH_MUS = np.asarray([0.25, 0.75, 1.0, 1.75, 2.50])
INVERSION_MU_POINTS = 401 if FAST_MODE else 1_001
INVERSION_MU_GRID = np.linspace(*MU_RANGE, INVERSION_MU_POINTS)
observed_rng = np.random.default_rng(SEED + 1600)
observed_mean = (
    OBSERVED_TRUTH_MUS[:, None]
    * LAM_SIG
    * SIM_AUDIT_SIGNAL_PROBABILITY[None, :]
    + LAM_BKG * SIM_AUDIT_BACKGROUND_PROBABILITY[None, :]
)
observed_counts = observed_rng.poisson(observed_mean)


def response_baseline_accepts_95(mu, t_mu):
    """Endpoint-aware response-matched hNDE rule before PIT calibration."""
    mu, t_mu = np.broadcast_arrays(
        np.asarray(mu, dtype=np.float64),
        np.asarray(t_mu, dtype=np.float64),
    )
    accepted = np.empty(mu.shape, dtype=bool)
    assigned = np.zeros(mu.shape, dtype=bool)
    interior = (mu > MU_RANGE[0]) & (mu < MU_RANGE[1])
    if np.any(interior):
        u0, _ = evaluate_calibrated_pit(mu[interior], t_mu[interior])
        accepted[interior] = u0 <= .95
        assigned[interior] = True
    for anchor_index, anchor_mu in enumerate(ANCHOR_MUS):
        endpoint = mu == anchor_mu
        accepted[endpoint] = (
            t_mu[endpoint]
            <= anchor_hnde_quantiles[anchor_index, index_95]
        )
        assigned[endpoint] = True
    if not np.all(assigned):
        raise ValueError("mu lies outside the calibrated design range.")
    return accepted


def confidence_set_components(grid, accepted):
    """Return midpoint-interpolated connected components of a grid set."""
    grid = np.asarray(grid, dtype=np.float64)
    accepted = np.asarray(accepted, dtype=bool)
    if grid.ndim != 1 or accepted.shape != grid.shape:
        raise ValueError("grid and accepted must be equal-length vectors.")
    padded = np.pad(accepted.astype(np.int8), (1, 1))
    changes = np.diff(padded)
    starts = np.flatnonzero(changes == 1)
    stops = np.flatnonzero(changes == -1) - 1
    components = []
    for start, stop in zip(starts, stops):
        left = (
            grid[0]
            if start == 0
            else 0.5 * (grid[start - 1] + grid[start])
        )
        right = (
            grid[-1]
            if stop == len(grid) - 1
            else 0.5 * (grid[stop] + grid[stop + 1])
        )
        components.append((float(left), float(right)))
    return components


def format_components(components):
    if not components:
        return "empty"
    return " ∪ ".join(
        f"[{left:.3f}, {right:.3f}]" for left, right in components
    )


inversion_rows = []
inversion_curves = []
for observed_index, (truth_mu, counts) in enumerate(zip(
    OBSERVED_TRUTH_MUS, observed_counts
)):
    repeated_counts = np.repeat(
        counts[None, :], INVERSION_MU_POINTS, axis=0
    )
    mu_hat_scan, t_scan, _ = response_corrected_test_statistic(
        repeated_counts, INVERSION_MU_GRID
    )
    scan_interior = (
        (INVERSION_MU_GRID > MU_RANGE[0])
        & (INVERSION_MU_GRID < MU_RANGE[1])
    )
    u0_scan = np.empty(INVERSION_MU_POINTS, dtype=np.float64)
    u_cal_scan = np.empty(INVERSION_MU_POINTS, dtype=np.float64)
    u0_scan[scan_interior], u_cal_scan[scan_interior] = (
        evaluate_calibrated_pit(
            INVERSION_MU_GRID[scan_interior],
            t_scan[scan_interior],
        )
    )
    # At the exact boundaries the construction uses empirical anchor
    # laws, so use their empirical CDFs for the displayed pivots too.
    for anchor_mu in ANCHOR_MUS:
        endpoint = INVERSION_MU_GRID == anchor_mu
        endpoint_t = float(t_scan[endpoint][0])
        u0_scan[endpoint] = np.mean(
            anchor_hnde[anchor_mu]["t_mu"] <= endpoint_t
        )
        u_cal_scan[endpoint] = np.mean(
            anchor_simulator[anchor_mu]["t_mu"] <= endpoint_t
        )
    accepted_before = response_baseline_accepts_95(
        INVERSION_MU_GRID, t_scan
    )
    accepted_after = neyman_accepts_95(
        INVERSION_MU_GRID, t_scan
    )
    components_before = confidence_set_components(
        INVERSION_MU_GRID, accepted_before
    )
    components_after = confidence_set_components(
        INVERSION_MU_GRID, accepted_after
    )
    length_before = sum(
        right - left for left, right in components_before
    )
    length_after = sum(
        right - left for left, right in components_after
    )
    inversion_rows.append({
        "dataset": observed_index,
        "truth_mu": float(truth_mu),
        "surrogate_mu_hat": float(np.median(mu_hat_scan)),
        "response_baseline_set": format_components(components_before),
        "calibrated_set": format_components(components_after),
        "response_baseline_length": float(length_before),
        "calibrated_length": float(length_after),
        "calibrated_minus_baseline": float(length_after - length_before),
    })
    inversion_curves.append({
        "u0": u0_scan,
        "u_cal": u_cal_scan,
        "before": accepted_before,
        "after": accepted_after,
    })

inversion_table = pd.DataFrame(inversion_rows)
display(inversion_table.style.format({
    "truth_mu": "{:.3f}",
    "surrogate_mu_hat": "{:.3f}",
    "response_baseline_length": "{:.3f}",
    "calibrated_length": "{:.3f}",
    "calibrated_minus_baseline": "{:+.3f}",
}).hide(axis="index"))
print(
    "Mean confidence-set length, response baseline / calibrated:",
    f"{inversion_table['response_baseline_length'].mean():.4f} / "
    f"{inversion_table['calibrated_length'].mean():.4f}",
)

fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.5), sharex=True, sharey=True)
for ax, truth_mu, curves in zip(
    axes.flat, OBSERVED_TRUTH_MUS, inversion_curves
):
    ax.plot(
        INVERSION_MU_GRID, curves["u0"], ls="--", lw=1.8,
        label="response baseline",
    )
    ax.plot(
        INVERSION_MU_GRID, curves["u_cal"], lw=2.0,
        label="simulator calibrated",
    )
    ax.axhline(.95, color="black", ls=":", lw=1.1)
    ax.axvline(truth_mu, color="C2", ls="-.", lw=1.1)
    ax.fill_between(
        INVERSION_MU_GRID, 0.0, 1.0,
        where=curves["after"], color="C0", alpha=.08,
    )
    ax.set_title(rf"pseudo-data truth $\mu={truth_mu:g}$")
    ax.grid(alpha=.2)
axes.flat[len(OBSERVED_TRUTH_MUS)].axis("off")
for ax in axes[-1, :]:
    ax.set_xlabel(r"tested physical $\mu$")
for ax in axes[:, 0]:
    ax.set_ylabel("conditional CDF / pivot")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Neyman inversion before and after residual simulator calibration")
fig.tight_layout()
export_exercise11_figure(fig, "response_pit_confidence_set_inversion")
plt.show()


## Interpretation: what each layer corrected

This exercise now separates effects that were previously entangled:

- The frozen hNDE likelihood supplies a computationally convenient ordering model in its own coordinate $\nu$.
- The streamed analytic simulator replaces a noisy 250k-event construction with a registered 64M-event-per-process law while keeping peak memory bounded.
- The nested 250k, 1M, 4M, 16M, and 64M prefixes reveal whether $q$ moments and the score-projected response actually converge; raw $L^1$ alone is not a sufficient diagnostic.
- The pseudo-truth map $g_{64\mathrm M}(\mu)$ removes the leading simulator-to-hNDE response displacement. It is a reparameterization of the tested hypothesis, not a claim that the learned likelihood became true.
- The response-matched hNDE spline and first ratio estimate the inexpensive reference CDF $F_{\rm H}^{(g)}$.
- The PIT ratio estimates residual simulator differences in variance, skewness, discreteness, and tails. It learns all confidence levels at once.
- The same-template internal audit isolates conditional-calibration error under the declared 64M construction law.
- The multinomial bootstrap measures sensitivity of the fixed construction to finite construction-template fluctuations.
- The newly seeded 64M audit tests transfer to simulator events that never entered Exercise 5 or the construction and were not exposed until the procedure was frozen.
- The post-unsealing fixed-$\mu$ study asks directly how coverage transfer changes with template size without retraining five neural pipelines.

The 95% confidence set for observed counts is obtained by evaluating the response-corrected statistic and inverting

$$
\mathcal C_{0.95}(\mathcal D_{\rm obs})
=\left\{\mu\in(0,3):
G\!\left(
F_{\rm H}^{(g)}(t_\mu^{(g)}(\mathcal D_{\rm obs})\mid\mu)
\mid\mu\right)
\leq0.95\right\},
$$

with explicit empirical rules at the two exact endpoints. The equivalent cutoff form is

$$
t_\mu^{(g)}(\mathcal D_{\rm obs})
\leq
\left(F_{\rm H}^{(g)}\right)^{-1}
\!\left(G^{-1}(0.95\mid\mu)\mid\mu\right).
$$

Calibration controls size; it does not guarantee power. A poor ordering model can have correct coverage yet give longer or asymmetric confidence sets. Response matching is useful precisely because it can improve the ordering before the residual size calibration is applied.


## What the large-template study does—and does not—establish

The notebook now reports four distinct empirical targets rather than calling all of them “coverage”:

1. **Generator-version closure.** The current Exercise 5 parquets and the fresh analytic stream are compared in PRESEL acceptance, selected-feature histograms, $q$ moments, and response.
2. **Convergence of the construction law.** Nested prefixes show $L^1$, $g_N-g_{64\mathrm M}$, and standardized score drift. Because prefixes are nested, their fluctuations are correlated.
3. **Conditional calibration on the 64M construction template.** The internal audit asks whether the learned map reproduces that one fixed empirical law.
4. **Transfer to a new 64M simulator law.** The primary audit and post-unsealing fixed-$\mu$ study use entirely new seeds and events.

Streaming 64M selected events per process greatly reduces the finite-shape-template error, but it does not make every frozen ingredient exact. Coverage remains conditional on the post-selection yields $\lambda_S$ and $\lambda_B$, the learned PRESEL and ratio networks, their finite-reference normalizations, the frozen hNDE likelihood, and the 512-bin compression. The existing compression error is checked separately and does not vanish by increasing the simulator reservoir.

The audit law is also an empirical approximation rather than an infinite simulator truth. Its role is to provide a high-precision, independently seeded stress test of transfer. If its result motivates another method change, that audit has been consumed and a new seed/reservoir is required before making another external coverage claim.

Other production targets remain possible:

- average coverage under an explicitly hierarchical template law;
- uniformly conservative coverage over a specified template nuisance set;
- conditional coverage after augmenting the calibrated parameter vector with MC-statistical or modeling nuisances.

Those are different ensembles and generally different confidence procedures. They should not be inferred from the diagnostic bootstrap used here.


## Why the PIT correction still extends to high-dimensional parameters

For a physical parameter vector $\boldsymbol\theta$, the pseudo-truth response becomes a map $\boldsymbol g(\boldsymbol\theta)$ into the surrogate-likelihood coordinates. The models are

$$
q_\phi(y\mid\boldsymbol\theta),
\qquad
r_1(y,\boldsymbol\theta),
\qquad
r_{\rm cal}(u,\boldsymbol\theta).
$$

The simulator classifier input has dimension $d+1$, but its conditional normalizer remains

$$
Z_{\rm cal}(\boldsymbol\theta)
=\int_0^1r_{\rm cal}(u,\boldsymbol\theta)\,du,
$$

a one-dimensional integral. The same is true for the $t$ primitive because the ordering statistic is scalar. In high dimensions one evaluates these integrals on demand for requested parameter batches rather than tabulating a Cartesian grid.

The response map itself becomes harder: it must be identifiable, sufficiently smooth, and locally one-to-one over the inference region if it is to serve as a useful reparameterization. Nuisance directions may be included when the desired guarantee is conditional on them, or marginalized according to a precisely stated ensemble when average coverage is the target.

PIT space solves the density-scale and normalization problem; it does not abolish the simulator curse of dimensionality. Proposal coverage, interpolation, event-level independence, and an external coverage auditor remain essential.


## Further studies

1. Compare the full PIT-density correction with direct 95% conditional quantile regression, the canonical LF2I branch. The ratio learns all confidence levels; a tail-specific regressor may be more efficient for one fixed level.
2. If the 64M prefixes have not stabilized in the standardized score direction, pre-register and launch a new 256M selected-background stream while retaining the registered 64M result as the pre-existing benchmark. The current exact-prefix cache intentionally stops at 64M.
3. Recompute $g(\mu)$ and the full calibration inside template bootstraps to study a hierarchical average-coverage construction, keeping that target distinct from robustness of the fixed construction.
4. Propagate uncertainty in $\lambda_S$, $\lambda_B$, finite-reference ratio normalization, and learned density ratios as explicit nuisance coordinates rather than conditioning on them.
5. Repeat the confidence-set-length comparison over a large pseudo-observed ensemble and study power, disconnected-set frequency, and design-range truncation.
6. If the newly seeded 64M audit motivates any method change, generate another independent simulator stream before making a fresh external coverage claim.
